In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path(r"C:\Users\Vivian\Documents\CONCH")
sys.path.insert(0, str(PROJECT_ROOT))

from conch.open_clip_custom import create_model_from_pretrained, tokenize, get_tokenizer
import torch
import os
from PIL import Image
from pathlib import Path

# show all jupyter output
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

c:\Users\Vivian\anaconda3\envs\conch\lib\site-packages\timm\models\layers\__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
model_cfg = 'conch_ViT-B-16'
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
# checkpoint_path = 'checkpoints/CONCH/pytorch_model.bin'
checkpoint_path = 'C:\\Users\\Vivian\\Documents\\CONCH\\checkpoints\\conch\\pytorch_model.bin' 
# checkpoint_path = r"C:\Users\Vivian\Documents\CONCH\_finetune_weights\fine_tuned_model2.pth" # testing with finetuned conch model
model, preprocess = create_model_from_pretrained(model_cfg, checkpoint_path, device=device)
_ = model.eval()

C:\Users\Vivian\Documents\CONCH\conch\open_clip_custom\factory.py:18: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location=map

In [3]:
# import torch
from transformers import AutoTokenizer
tokenizer = get_tokenizer()



In [3]:
PROMPTS = {
    # ------------------------------------------------------------
    # Class-level prompt templates (use multiple templates; average)
    # ------------------------------------------------------------
    "templates": [
        "breast histopathology image of {}",
        "H&E stained breast tissue showing {}",
        "breast core biopsy with features of {}",
        "microscopy of breast lesion consistent with {}",
    ],

    "classes": {
        "FA": [
            "fibroadenoma",
            "fibroadenoma of the breast",
            "benign fibroepithelial lesion (fibroadenoma)",
        ],
        "PT": [
            "phyllodes tumor",
            "phyllodes tumour",
            "fibroepithelial lesion consistent with phyllodes tumor",
        ]
        # ,
        # "NORMAL": [
        #     "normal breast tissue",
        #     "benign breast tissue",
        #     "background breast parenchyma",
        # ],
    },

    # ------------------------------------------------------------
    # Mechanism / diagnostic-feature concepts (graded where helpful)
    # These are the ones you can report as explanations too.
    # ------------------------------------------------------------
    "concepts": {
        # ---- FA-lean concepts (benign, bland) ----
        "FA_lean": [
            "bland fibrous stroma",
            "low stromal cellularity",
            "uniform stromal nuclei",
            "evenly distributed chromatin",
            "inconspicuous nucleoli in stromal cells",
            "balanced glands and stroma",
            "benign breast ducts within fibrous stroma",
            "pericanalicular pattern",
            "intracanalicular pattern",
            # allowed but non-worrisome
            "multinucleated stromal giant cells",
            "mild fragmentation with low cellularity",
            "edematous stroma",
            "myxoid stroma",
            "smooth muscle stromal metaplasia",
        ],

        # ---- PT-lean concepts (suspicious) ----
        "PT_lean": [
            # stromal hypercellularity / condensation
            "mild stromal hypercellularity",
            "moderate stromal hypercellularity",
            "marked stromal hypercellularity",
            "periepithelial stromal condensation",
            # atypia spectrum
            "mild stromal cell atypia",
            "moderate stromal cell atypia",
            "marked stromal cell atypia",
            "stromal nuclear hyperchromasia",
            "irregular nuclear membranes in stromal cells",
            "prominent nucleoli in stromal cells",
            "stromal nuclear pleomorphism",
            # fragmentation scenario
            "highly fragmented hypercellular specimen with epithelium at the ends",
            # architecture
            "leaf-like architecture",
            "leaf-like stromal projections",
            "intralesional heterogeneity in fibroepithelial lesion",
            # border
            "infiltrative or permeative lesion border",
            "entrapped fat cells at lesion border",
        ],

        # ---- PT grade cues (optional separate analysis) ----
        "PT_benign_like": [
            "mild stromal hypercellularity",
            "mild stromal cell atypia",
            "early leaf-like architecture",
            "periepithelial stromal condensation",
        ],
        "PT_borderline_like": [
            "moderate stromal hypercellularity",
            "moderate stromal cell atypia",
            "three or more mitoses per ten high power fields",
            "fragmented hypercellular stromal tissue",
            "heterogeneous stromal architecture",
        ],
        "PT_malignant_like": [
            "stromal overgrowth with absent epithelium",
            "sheets of stroma without glandular elements",
            "marked stromal cell atypia",
            "five or more mitoses per ten high power fields",
            "tumor necrosis with karyorrhexis",
        ],

        # ---- mitotic concepts (if you want explicit prompts) ----
        "mitosis": [
            "stromal mitotic figures",
            "mitotic hotspot in hypercellular stroma",
            "three or more mitoses per ten high power fields",
            "five or more mitoses per ten high power fields",
        ],

        # ---- normal/background concepts ----
        "NORMAL_lean": [
            "normal breast lobules",
            "benign ducts and adipose tissue",
            "background breast parenchyma",
            "non-neoplastic breast stroma",
            "fibroadipose tissue",
            "fat lobules",
        ],

        # ---- non-diagnostic / negative controls ----
        "controls": [
            "tissue section edge",
            "low-information fibrous tissue",
            "out of focus histology",
        ],
    },
}


In [4]:
def build_prompt_sets(PROMPTS):
    templates = PROMPTS["templates"]

    # class prompts
    class_prompts = {}
    for cls_key, synonyms in PROMPTS["classes"].items():
        plist = []
        for s in synonyms:
            plist.extend([t.format(s) for t in templates])
        class_prompts[cls_key] = plist

    # concept prompts
    concept_prompts = PROMPTS["concepts"]

    return class_prompts, concept_prompts


full script

In [17]:
"""
CONCH zero-shot slide scoring from .npy patch directories using FULL PROMPTS dict:
- Produces BOTH:
  (A) class-prompt prediction (FA/PT/NORMAL) using templates × class synonyms
  (B) concept-bucket scores using PROMPTS["concepts"] (FA_lean, PT_lean, NORMAL_lean, etc.)

Outputs 2 CSVs:
  1) per-slide class probabilities + bucket means + metadata
  2) (optional) per-slide top concepts (mean similarity across patches) for interpretability

Notes:
- Your dataset folders may only contain FA/PT slides; NORMAL may not exist as slides, but the
  NORMAL prompts can still be used as a "background" class in zero-shot scoring.
- This script is meant to be run in a notebook OR a python script where `model`, `preprocess`,
  and `tokenizer` are already defined (as in your current setup).

Folder layout assumed:
PATCH_ROOT/
  20x/
    FA/<slide_id>/*.npy
    PT/<slide_id>/*.npy
  40x/
    FA/<slide_id>/*.npy
    PT/<slide_id>/*.npy
"""

import os
import glob
import argparse
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn.functional as F
from PIL import Image


# -------------------------
# FULL PROMPTS (paste yours here)
# -------------------------
PROMPTS = {
    "templates": [
        "breast histopathology image of {}",
        "H&E stained breast tissue showing {}",
        "breast core biopsy with features of {}",
        "microscopy of breast lesion consistent with {}",
    ],
    "classes": {
        "FA": [
            "fibroadenoma",
            "fibroadenoma of the breast",
            "benign fibroepithelial lesion (fibroadenoma)",
        ],
        "PT": [
            "phyllodes tumor",
            "phyllodes tumour",
            "fibroepithelial lesion consistent with phyllodes tumor",
        ]
        # ,
        # "NORMAL": [
        #     "normal breast tissue",
        #     "benign breast tissue",
        #     "background breast parenchyma",
        # ],
    },
    "concepts": {
        "FA_lean": [
            "bland fibrous stroma",
            "low stromal cellularity",
            "uniform stromal nuclei",
            "evenly distributed chromatin",
            "inconspicuous nucleoli in stromal cells",
            "balanced glands and stroma",
            "benign breast ducts within fibrous stroma",
            "pericanalicular pattern",
            "intracanalicular pattern",
            "multinucleated stromal giant cells",
            "mild fragmentation with low cellularity",
            "edematous stroma",
            "myxoid stroma",
            "smooth muscle stromal metaplasia",
        ],
        "PT_lean": [
            "mild stromal hypercellularity",
            "moderate stromal hypercellularity",
            "marked stromal hypercellularity",
            "periepithelial stromal condensation",
            "mild stromal cell atypia",
            "moderate stromal cell atypia",
            "marked stromal cell atypia",
            "stromal nuclear hyperchromasia",
            "irregular nuclear membranes in stromal cells",
            "prominent nucleoli in stromal cells",
            "stromal nuclear pleomorphism",
            "highly fragmented hypercellular specimen with epithelium at the ends",
            "leaf-like architecture",
            "leaf-like stromal projections",
            "intralesional heterogeneity in fibroepithelial lesion",
            "infiltrative or permeative lesion border",
            "entrapped fat cells at lesion border",
        ],
        "PT_benign_like": [
            "mild stromal hypercellularity",
            "mild stromal cell atypia",
            "early leaf-like architecture",
            "periepithelial stromal condensation",
        ],
        "PT_borderline_like": [
            "moderate stromal hypercellularity",
            "moderate stromal cell atypia",
            "three or more mitoses per ten high power fields",
            "fragmented hypercellular stromal tissue",
            "heterogeneous stromal architecture",
        ],
        "PT_malignant_like": [
            "stromal overgrowth with absent epithelium",
            "sheets of stroma without glandular elements",
            "marked stromal cell atypia",
            "five or more mitoses per ten high power fields",
            "tumor necrosis with karyorrhexis",
        ],
        "mitosis": [
            "stromal mitotic figures",
            "mitotic hotspot in hypercellular stroma",
            "three or more mitoses per ten high power fields",
            "five or more mitoses per ten high power fields",
        ],
        "NORMAL_lean": [
            "normal breast lobules",
            "benign ducts and adipose tissue",
            "background breast parenchyma",
            "non-neoplastic breast stroma",
            "fibroadipose tissue",
            "fat lobules",
        ],
        "controls": [
            "tissue section edge",
            "low-information fibrous tissue",
            "out of focus histology",
        ],
    },
}


# -------------------------
# NPY -> PIL
# -------------------------
def load_npy_as_pil(npy_path: str) -> Image.Image:
    arr = np.load(npy_path)
    if arr.dtype != np.uint8:
        arr = np.clip(arr, 0, 255).astype(np.uint8)
    if arr.ndim == 2:
        arr = np.stack([arr] * 3, axis=-1)
    if arr.ndim == 3 and arr.shape[-1] == 4:
        arr = arr[..., :3]
    return Image.fromarray(arr)


# -------------------------
# Text encoding (matches your working signature)
# -------------------------
def encode_text_prompts(prompts: List[str], tokenizer, model, device: str) -> np.ndarray:
    enc = tokenizer(prompts, padding=True, return_tensors="pt")
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.inference_mode():
        txt = model.encode_text(enc["input_ids"])
        txt = F.normalize(txt, dim=-1)
    return txt.detach().cpu().numpy()


# -------------------------
# Image encoding
# -------------------------
def encode_patch_embedding(npy_path: str, preprocess, model, device: str) -> np.ndarray:
    img = load_npy_as_pil(npy_path)
    x = preprocess(img).unsqueeze(0).to(device)
    with torch.inference_mode():
        e = model.encode_image(x)
        e = F.normalize(e, dim=-1)
    return e.detach().cpu().numpy()[0]


# -------------------------
# Prompt-bank builders
# -------------------------
def build_flat_class_prompts(P: dict) -> Tuple[List[str], np.ndarray, List[str]]:
    """
    templates × class synonyms -> flat prompts + class index map
    """
    templates = P["templates"]
    classes = P["classes"]
    class_names = list(classes.keys())  # ["FA","PT","NORMAL"] in your dict

    flat_prompts: List[str] = []
    flat_class_map: List[int] = []

    for ci, cls in enumerate(class_names):
        for syn in classes[cls]:
            for t in templates:
                flat_prompts.append(t.format(syn))
                flat_class_map.append(ci)

    return flat_prompts, np.array(flat_class_map, dtype=int), class_names


def build_flat_concepts(P: dict) -> Tuple[List[str], List[str], Dict[str, List[int]]]:
    """
    Flatten all concept prompts; also return indices per concept group.
    Dedupe while preserving order (important).
    """
    concept_list: List[str] = []
    concept_group: List[str] = []
    for grp, plist in P["concepts"].items():
        for p in plist:
            concept_list.append(p)
            concept_group.append(grp)

    # dedupe preserving order
    seen = set()
    out_list, out_group = [], []
    for p, g in zip(concept_list, concept_group):
        if p not in seen:
            out_list.append(p)
            out_group.append(g)
            seen.add(p)

    concept_list = out_list
    concept_group = out_group

    group_to_indices: Dict[str, List[int]] = {}
    for i, g in enumerate(concept_group):
        group_to_indices.setdefault(g, []).append(i)

    return concept_list, concept_group, group_to_indices


# -------------------------
# Patch scoring
# -------------------------
def softmax_np(x: np.ndarray) -> np.ndarray:
    x = x - np.max(x)
    e = np.exp(x)
    return e / (e.sum() + 1e-12)


def patch_class_probs(img_emb: np.ndarray, text_emb_class: np.ndarray, flat_class_map: np.ndarray, n_classes: int) -> np.ndarray:
    """
    For stability:
      sims -> softmax across ALL prompts -> average mass per class -> normalize
    """
    sims = img_emb @ text_emb_class.T  # (M,)
    p_prompt = softmax_np(sims)        # (M,)

    out = np.zeros(n_classes, dtype=float)
    for c in range(n_classes):
        out[c] = p_prompt[flat_class_map == c].mean()
    out = out / (out.sum() + 1e-12)
    return out  # (C,)


def patch_concept_sims(img_emb: np.ndarray, text_emb_concepts: np.ndarray) -> np.ndarray:
    """
    Raw cosine similarity per concept prompt
    """
    return img_emb @ text_emb_concepts.T  # (Nconcept,)


# -------------------------
# Slide scoring (aggregate patches)
# -------------------------
def classify_slide_dir(
    slide_dir: str,
    preprocess,
    model,
    device: str,
    text_emb_class: np.ndarray,
    flat_class_map: np.ndarray,
    class_names: List[str],
    text_emb_concepts: np.ndarray,
    group_to_indices: Dict[str, List[int]],
    max_patches: Optional[int] = 800,
    topk_patches: Optional[int] = 200,

    select_mode: str = "conf",          # NEW
    select_bucket: str = "PT_lean",     # NEW (only used if select_mode="concept")
    seed: int = 0,                      # NEW (for random sampling)
) -> Dict[str, object]:
    paths = sorted(glob.glob(os.path.join(slide_dir, "*.npy")))
    if len(paths) == 0:
        raise RuntimeError(f"No .npy patches found in: {slide_dir}")

    # random sample N patches (instead of first N sorted)
    if max_patches is not None and len(paths) > max_patches:
        rng = np.random.default_rng(seed)  # seed passed into function
        paths = rng.choice(paths, size=max_patches, replace=False).tolist()

    # (optional) sort only for reproducible iteration order after sampling
    paths = sorted(paths)    

    # if max_patches is not None and len(paths) > max_patches:
    #     paths = paths[:max_patches]

    patch_probs = []
    patch_conf = []
    patch_concept = []

    n_classes = len(class_names)

    patch_select_score = []

    for pth in paths:
        img_emb = encode_patch_embedding(pth, preprocess, model, device)

        probs = patch_class_probs(img_emb, text_emb_class, flat_class_map, n_classes)
        patch_probs.append(probs)
        patch_conf.append(float(probs.max()))

        cs = patch_concept_sims(img_emb, text_emb_concepts)
        patch_concept.append(cs)

        # NEW: selection score
        if select_mode == "conf":
            patch_select_score.append(float(probs.max()))
        elif select_mode == "concept":
            inds = group_to_indices.get(select_bucket, [])
            if len(inds) == 0:
                raise ValueError(f"Unknown/empty bucket for selection: {select_bucket}")
            patch_select_score.append(float(np.mean(cs[inds])))
        else:
            raise ValueError(f"Bad select_mode: {select_mode}")
        # --------------

    patch_probs = np.stack(patch_probs, axis=0)       # (N, C)
    patch_conf  = np.array(patch_conf, dtype=float)   # (N,)
    patch_concept = np.stack(patch_concept, axis=0)   # (N, Nconcept)
    patch_select_score = np.array(patch_select_score, dtype=float)


    # top-k pooling by confidence to reduce background dominance
    if topk_patches is not None and topk_patches < patch_probs.shape[0]:
        # idx = np.argsort(-patch_conf)[:topk_patches]
        idx = np.argsort(-patch_select_score)[:topk_patches]

        probs_used = patch_probs[idx]
        concept_used = patch_concept[idx]
        conf_used = patch_conf[idx]
    else:
        idx = np.arange(patch_probs.shape[0])
        probs_used = patch_probs
        concept_used = patch_concept
        conf_used = patch_conf

    slide_probs = probs_used.mean(axis=0)  # (C,)
    pred_idx = int(np.argmax(slide_probs))
    pred_name = class_names[pred_idx]
    margin = float(np.sort(slide_probs)[-1] - np.sort(slide_probs)[-2]) if len(slide_probs) >= 2 else np.nan

    # bucket means over concepts: mean similarity per concept prompt, then average within group
    mean_concept = concept_used.mean(axis=0)  # (Nconcept,)
    
    bucket_scores = {}
    for grp, inds in group_to_indices.items():
        bucket_scores[f"bucket_{grp}"] = float(np.mean(mean_concept[inds])) if len(inds) else np.nan

    # top concepts overall for interpretability (by mean similarity)
    topc_idx = np.argsort(-mean_concept)[:15]
    top_concepts = [(int(i), float(mean_concept[i])) for i in topc_idx]

    return {
        "slide_probs": slide_probs,
        "pred_idx": pred_idx,
        "pred_name": pred_name,
        "margin": margin,
        "n_patches_scored": int(len(paths)),
        "n_patches_used": int(len(idx)),
        "bucket_scores": bucket_scores,
        "top_concepts": top_concepts,
        "mean_concept": mean_concept,

    }


# -------------------------
# Dataset scanning
# -------------------------
def list_slide_dirs(mag_root: str) -> List[Tuple[str, str]]:
    out = []
    for cls in ["FA", "PT", "NORMAL"]:  # NORMAL folder may not exist; we just skip if missing
        cls_root = os.path.join(mag_root, cls)
        if not os.path.isdir(cls_root):
            continue
        for slide_id in sorted(os.listdir(cls_root)):
            slide_dir = os.path.join(cls_root, slide_id)
            if os.path.isdir(slide_dir):
                out.append((slide_id, slide_dir))
    return out


def infer_label_from_path(slide_dir: str) -> str:
    # label string from parent folder name
    return os.path.basename(os.path.dirname(slide_dir)).upper()


# -------------------------
# MAIN
# -------------------------
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--patch_root",
        type=str,
        default=r"C:\Users\Vivian\Documents\CONCH\all_patches\patches_2.5x",
        help="Root containing 20x/40x subdirs.",
    )
    parser.add_argument("--mag", type=str, default="40x", choices=["20x", "40x"])
    parser.add_argument(
        "--out_csv",
        type=str,
        default=r"C:\Users\Vivian\Documents\CONCH\test_text_encoder\2.5x_noNORM_conch_zeroshot_slides_fullprompts_40x.csv",
    )
    parser.add_argument(
        "--out_top_concepts_csv",
        type=str,
        default=r"C:\Users\Vivian\Documents\CONCH\test_text_encoder\2.5x_noNORM_conch_zeroshot_slides_top_concepts_40x.csv",
        help="Optional: per-slide top concepts (mean similarity) saved here.",
    )
    parser.add_argument("--max_patches", type=int, default=800)
    parser.add_argument("--topk_patches", type=int, default=200)
    parser.add_argument("--device", type=str, default="cuda" if torch.cuda.is_available() else "cpu")
    parser.add_argument("--seed", type=int, default=0)
    parser.add_argument("--select_mode", type=str, default="conf", choices=["conf", "concept"])
    parser.add_argument("--select_bucket", type=str, default="PT_lean")

    # args = parser.parse_args()
    args, _ = parser.parse_known_args()


    # expects these already exist in your environment
    global model, preprocess, tokenizer
    if "model" not in globals() or "preprocess" not in globals() or "tokenizer" not in globals():
        raise RuntimeError(
            "Expected `model`, `preprocess`, `tokenizer` to be defined already.\n"
            "Run this in your notebook, or paste your CONCH loading code above main()."
        )

    device = args.device
    model.to(device)
    model.eval()

    mag_root = os.path.join(args.patch_root, args.mag)
    if not os.path.isdir(mag_root):
        raise RuntimeError(f"Mag root not found: {mag_root}")

    # Build + encode class prompts
    flat_class_prompts, flat_class_map, class_names = build_flat_class_prompts(PROMPTS)
    text_emb_class = encode_text_prompts(flat_class_prompts, tokenizer, model, device)

    # Build + encode concept prompts
    concept_list, concept_group, group_to_indices = build_flat_concepts(PROMPTS)
    text_emb_concepts = encode_text_prompts(concept_list, tokenizer, model, device)

    # Scan slide directories
    slides = list_slide_dirs(mag_root)
    print(f"Found {len(slides)} slide folders under {mag_root}")

    rows = []
    top_rows = []

    for slide_id, slide_dir in tqdm(slides, desc=f"Scoring slides ({args.mag})"):
        true_label_str = infer_label_from_path(slide_dir)  # "FA" or "PT" or "NORMAL"

        res = classify_slide_dir(
            slide_dir=slide_dir,
            preprocess=preprocess,
            model=model,
            device=device,
            text_emb_class=text_emb_class,
            flat_class_map=flat_class_map,
            class_names=class_names,
            text_emb_concepts=text_emb_concepts,
            group_to_indices=group_to_indices,
            max_patches=args.max_patches if args.max_patches > 0 else None,
            topk_patches=args.topk_patches if args.topk_patches > 0 else None,
            select_mode=args.select_mode,
            select_bucket=args.select_bucket,
            seed=args.seed,
        )

        slide_probs = res["slide_probs"]
        pred_name = res["pred_name"]

        row = {
            "mag": args.mag,
            "slide_id": slide_id,
            "true_label_str": true_label_str,
            "pred_class": pred_name,
            "margin": res["margin"],
            "n_patches_scored": res["n_patches_scored"],
            "n_patches_used": res["n_patches_used"],
            "slide_dir": slide_dir,
        }

        # add class probs
        for i, cls in enumerate(class_names):
            row[f"prob_{cls.lower()}"] = float(slide_probs[i])

        # add bucket scores
        row.update(res["bucket_scores"])

        for i, cname in enumerate(concept_list):
            row[f"concept_{cname}"] = float(res["mean_concept"][i])


        rows.append(row)

        # record top concepts
        for rank, (ci, score) in enumerate(res["top_concepts"], start=1):
            top_rows.append({
                "mag": args.mag,
                "slide_id": slide_id,
                "true_label_str": true_label_str,
                "pred_class": pred_name,
                "rank": rank,
                "concept_group": concept_group[ci],
                "concept": concept_list[ci],
                "mean_similarity": score,
            })

    df = pd.DataFrame(rows)
    df.to_csv(args.out_csv, index=False)
    print("Saved:", args.out_csv)

    df_top = pd.DataFrame(top_rows)
    df_top.to_csv(args.out_top_concepts_csv, index=False)
    print("Saved:", args.out_top_concepts_csv)

    # Quick sanity summary (not CV-controlled)
    if len(df) > 0:
        # only compute "accuracy" where true_label_str is one of the model classes
        mask = df["true_label_str"].isin(class_names)
        if mask.any():
            acc = (df.loc[mask, "pred_class"].values == df.loc[mask, "true_label_str"].values).mean()
            print(f"\nNaive accuracy on this directory (not CV-controlled): {acc:.3f}")

        print("\nPrediction counts:")
        print(df["pred_class"].value_counts())

        print("\nMean bucket scores by predicted class:")
        bucket_cols = [c for c in df.columns if c.startswith("bucket_")]
        if bucket_cols:
            print(df.groupby("pred_class")[bucket_cols].mean())



'\nCONCH zero-shot slide scoring from .npy patch directories using FULL PROMPTS dict:\n- Produces BOTH:\n  (A) class-prompt prediction (FA/PT/NORMAL) using templates × class synonyms\n  (B) concept-bucket scores using PROMPTS["concepts"] (FA_lean, PT_lean, NORMAL_lean, etc.)\n\nOutputs 2 CSVs:\n  1) per-slide class probabilities + bucket means + metadata\n  2) (optional) per-slide top concepts (mean similarity across patches) for interpretability\n\nNotes:\n- Your dataset folders may only contain FA/PT slides; NORMAL may not exist as slides, but the\n  NORMAL prompts can still be used as a "background" class in zero-shot scoring.\n- This script is meant to be run in a notebook OR a python script where `model`, `preprocess`,\n  and `tokenizer` are already defined (as in your current setup).\n\nFolder layout assumed:\nPATCH_ROOT/\n  20x/\n    FA/<slide_id>/*.npy\n    PT/<slide_id>/*.npy\n  40x/\n    FA/<slide_id>/*.npy\n    PT/<slide_id>/*.npy\n'

In [15]:
import os
import pandas as pd
import numpy as np

def summarize_mag_csv(csv_path: str) -> pd.DataFrame:
    """
    Reads one of your generated slide-level CSVs and returns a 1-row summary:
    | Mag | Naive Acc | % PT preds | mean PT_lean | mean mitosis | mean NORMAL_lean |
    """
    df = pd.read_csv(csv_path)

    topk = None
    base = os.path.basename(csv_path).lower()
    if 'topk25' in base:
        topk = 25
    elif 'topk50' in base:
        topk = 50
    elif 'topk100' in base:
        topk = 100
    elif 'topk200' in base:
        topk = 200
    else:        topk = None
    # # --- infer mag label ---
    # mag = None
    # if "mag" in df.columns and df["mag"].nunique() == 1:
    #     mag = str(df["mag"].iloc[0])
    # else:
    #     # fallback: try to guess from filename
    #     base = os.path.basename(csv_path).lower()
    #     if "5x" in base: mag = "5x"
    #     elif "10x" in base: mag = "10x"
    #     elif "20x" in base: mag = "20x"
    #     elif "40x" in base: mag = "40x"
    #     else: mag = "unknown"

    # --- label/pred columns ---
    # Your script wrote: true_label_str and pred_class
    if "true_label_str" in df.columns:
        y_true = df["true_label_str"].astype(str).str.upper()
    elif "label" in df.columns:
        # numeric label fallback (0=FA,1=PT,2=NORMAL if you used that)
        y_true = df["label"].map({0:"FA", 1:"PT", 2:"NORMAL"}).fillna(df["label"].astype(str))
    else:
        raise ValueError(f"Couldn't find a true label column in {csv_path}")

    if "pred_class" in df.columns:
        y_pred = df["pred_class"].astype(str).str.upper()
    elif "pred" in df.columns:
        y_pred = df["pred"].map({0:"FA", 1:"PT", 2:"NORMAL"}).fillna(df["pred"].astype(str))
    else:
        raise ValueError(f"Couldn't find a prediction column in {csv_path}")

    # --- naive accuracy (only where true label is one of the predicted classes) ---
    valid = y_true.isin(["FA", "PT", "NORMAL"])
    naive_acc = float((y_true[valid].values == y_pred[valid].values).mean()) if valid.any() else np.nan

    # --- % PT predictions ---
    pct_pt = float((y_pred == "PT").mean())

    # --- bucket columns (may or may not exist depending on your CSV) ---
    def safe_mean(colname: str) -> float:
        return float(df[colname].mean()) if colname in df.columns else np.nan

    mean_pt_lean = safe_mean("bucket_PT_lean")
    mean_mitosis = safe_mean("bucket_mitosis")
    mean_normal_lean = safe_mean("bucket_NORMAL_lean")

    out = pd.DataFrame([{
        # "Mag": mag,
        'topk': topk,
        "Naive Acc": naive_acc,
        "% PT preds": pct_pt,
        "mean PT_lean": mean_pt_lean,
        "mean mitosis": mean_mitosis,
        "mean NORMAL_lean": mean_normal_lean,
        "n_slides": int(len(df)),
    }])
    return out


# -------------------------
# USAGE: add your CSV paths
# -------------------------
csv_paths = [
    r"C:\Users\Vivian\Documents\CONCH\test_text_encoder\concept_scores_v2\20x_randN800_topk200_conf.csv",
    r"C:\Users\Vivian\Documents\CONCH\test_text_encoder\concept_scores_v2\20x_randN800_topk100_conf.csv",
    r"C:\Users\Vivian\Documents\CONCH\test_text_encoder\concept_scores_v2\20x_randN800_topk50_conf.csv",
    r"C:\Users\Vivian\Documents\CONCH\test_text_encoder\concept_scores_v2\20x_randN800_topk25_conf.csv",
    r"C:\Users\Vivian\Documents\CONCH\test_text_encoder\concept_scores_v2\20x_randN800_topk0_conf.csv",
]

summaries = []
for p in csv_paths:
    if os.path.exists(p):
        summaries.append(summarize_mag_csv(p))
    else:
        print("Missing:", p)

summary_df = pd.concat(summaries, ignore_index=True)

# pretty formatting
summary_df["Naive Acc"] = summary_df["Naive Acc"].map(lambda x: f"{x:.3f}" if pd.notnull(x) else "nan")
summary_df["% PT preds"] = summary_df["% PT preds"].map(lambda x: f"{100*x:.1f}%" if pd.notnull(x) else "nan")
for c in ["mean PT_lean", "mean mitosis", "mean NORMAL_lean"]:
    summary_df[c] = summary_df[c].map(lambda x: f"{x:.3f}" if pd.notnull(x) else "nan")

summary_df


,topk,Naive Acc,% PT preds,mean PT_lean,mean mitosis,mean NORMAL_lean,n_slides
0,200,0.727,30.3%,0.180,0.125,0.208,33
1,100,0.758,27.3%,0.175,0.119,0.215,33
2,50,0.818,27.3%,0.170,0.114,0.219,33
3,25,0.788,24.2%,0.167,0.108,0.217,33
4,None,0.758,27.3%,0.181,0.124,0.188,33


In [ ]:
# for k in [25, 50, 100, 200]:
import sys
sys.argv = [
    "notebook",
    "--patch_root", r"C:\Users\Vivian\Documents\CONCH\patches_tiled\patches_10x",
    "--mag", "40x",
    "--out_csv", rf"C:\Users\Vivian\Documents\CONCH\test_text_encoder\concept_scores_v2\concept_score_40x_randN800_notopk_conf.csv",
    "--out_top_concepts_csv", rf"C:\Users\Vivian\Documents\CONCH\test_text_encoder\concept_scores_v2\concept_score_40x_randN800_notopk_conf_topconcepts.csv",
    "--max_patches", '800',
    "--topk_patches", 0,
    "--select_mode", "conf",
    "--seed", "0",
]
main()


load conch feats and align with concepts instead of encoding them each time

In [5]:
# OG CONCH FULL PROMPTS (paste yours here)
# -------------------------
PROMPTS = {
    "templates": [
        "breast histopathology image of {}",
        "H&E stained breast tissue showing {}",
        "breast core biopsy with features of {}",
        "microscopy of breast lesion consistent with {}",
    ],
    "classes": {
        "FA": [
            "fibroadenoma",
            "fibroadenoma of the breast",
            "benign fibroepithelial lesion (fibroadenoma)",
        ],
        "PT": [
            "phyllodes tumor",
            "phyllodes tumour",
            "fibroepithelial lesion consistent with phyllodes tumor",
        ]
        # ,
        # "NORMAL": [
        #     "normal breast tissue",
        #     "benign breast tissue",
        #     "background breast parenchyma",
        # ],
    },
    "concepts": {
        "FA_lean": [
            "bland fibrous stroma",
            "low stromal cellularity",
            "uniform stromal nuclei",
            "evenly distributed chromatin",
            "inconspicuous nucleoli in stromal cells",
            "balanced glands and stroma",
            "benign breast ducts within fibrous stroma",
            "pericanalicular pattern",
            "intracanalicular pattern",
            "multinucleated stromal giant cells",
            "mild fragmentation with low cellularity",
            "edematous stroma",
            "myxoid stroma",
            "smooth muscle stromal metaplasia",
        ],
        "PT_lean": [
            "mild stromal hypercellularity",
            "moderate stromal hypercellularity",
            "marked stromal hypercellularity",
            "periepithelial stromal condensation",
            "mild stromal cell atypia",
            "moderate stromal cell atypia",
            "marked stromal cell atypia",
            "stromal nuclear hyperchromasia",
            "irregular nuclear membranes in stromal cells",
            "prominent nucleoli in stromal cells",
            "stromal nuclear pleomorphism",
            "highly fragmented hypercellular specimen with epithelium at the ends",
            "leaf-like architecture",
            "leaf-like stromal projections",
            "intralesional heterogeneity in fibroepithelial lesion",
            "infiltrative or permeative lesion border",
            "entrapped fat cells at lesion border",
        ],
        "PT_benign_like": [
            "mild stromal hypercellularity",
            "mild stromal cell atypia",
            "early leaf-like architecture",
            "periepithelial stromal condensation",
        ],
        "PT_borderline_like": [
            "moderate stromal hypercellularity",
            "moderate stromal cell atypia",
            "three or more mitoses per ten high power fields",
            "fragmented hypercellular stromal tissue",
            "heterogeneous stromal architecture",
        ],
        "PT_malignant_like": [
            "stromal overgrowth with absent epithelium",
            "sheets of stroma without glandular elements",
            "marked stromal cell atypia",
            "five or more mitoses per ten high power fields",
            "tumor necrosis with karyorrhexis",
        ],
        "mitosis": [
            "stromal mitotic figures",
            "mitotic hotspot in hypercellular stroma",
            "three or more mitoses per ten high power fields",
            "five or more mitoses per ten high power fields",
        ],
        "NORMAL_lean": [
            "normal breast lobules",
            "benign ducts and adipose tissue",
            "background breast parenchyma",
            "non-neoplastic breast stroma",
            "fibroadipose tissue",
            "fat lobules",
        ],
        "controls": [
            "tissue section edge",
            "low-information fibrous tissue",
            "out of focus histology",
        ],
    },
}


In [130]:
# (BLOCK) SWAP IN GECKO CONCEPTS (description-only)
# Put this RIGHT AFTER your PROMPTS definition.
# =========================

GECKO_PROMPTS = {
  "FA": {
    "Low Stromal Cellularity": "Sparse stromal cells; low stromal cellularity; abundant collagen; absence of crowded stromal regions",
    "Balanced Glands and Stroma": "Balanced epithelial and stromal components; ducts embedded within fibrous stroma; biphasic lesion without stromal dominance",
    "Benign Breast Ducts Within Fibrous Stroma": "Benign ducts and lobules present within fibrous stroma; preserved ductal morphology; non-atypical epithelium",
    "Pericanalicular Pattern": "Pericanalicular architecture with stroma cuffing round ducts; organized benign fibroepithelial growth pattern",
    "Intracanalicular Pattern": "Intracanalicular pattern with compressed elongated ducts; stromal expansion into ductal spaces; benign fibroepithelial architecture",
    "Bland Fibrous Stroma": "Bland fibrous stroma; uniform collagenous background; non-hypercellular stromal appearance",
    "Uniform Stromal Nuclei": "Uniform stromal nuclei; bland nuclear appearance; minimal pleomorphism; cytologically benign stromal cells",
    "Evenly Distributed Chromatin": "Even chromatin distribution in stromal nuclei; absence of hyperchromasia; bland nuclear texture",
    "Inconspicuous Nucleoli in Stromal Cells": "Inconspicuous nucleoli; bland stromal cytology; no prominent nucleoli suggestive of atypia",
    "Well-Circumscribed Pushing Border": "Well-demarcated lesion with smooth rounded contours; pushing border; absence of infiltrative stromal tongues"
  },
  "PT": {
    "Mild Stromal Hypercellularity": "Mild increase in stromal cellularity; crowded stromal cells compared with fibroadenoma",
    "Moderate Stromal Hypercellularity": "Moderate stromal hypercellularity; dense stromal spindle cells; stromal prominence",
    "Marked Stromal Hypercellularity": "Marked stromal hypercellularity; very crowded stromal cells; stromal-dominant areas",
    "Periepithelial Stromal Condensation": "Stromal condensation around ducts; increased subepithelial stromal cellularity; cuffing pattern",
    "Stromal Nuclear Hyperchromasia": "Hyperchromatic stromal nuclei; dark irregular nuclear staining; increased atypia",
    "Stromal Nuclear Pleomorphism": "Variation in stromal nuclear size and shape; pleomorphism consistent with phyllodes",
    "Leaf-Like Architecture": "Leaf-like architecture with epithelial-lined clefts; fronded stromal projections",
    "Stromal Overgrowth Without Epithelium": "Broad sheets of stroma without glandular elements; pure stromal fields",
    "Stromal Mitotic Figures": "Mitotic figures within stromal cells; visible cell division in hypercellular stroma",
    "Mitotic Hotspot in Hypercellular Stroma": "Focal clusters of mitoses; concentrated mitotic activity within dense stromal regions"
  }
}

# overwrite your concept groups with GECKO-based buckets
PROMPTS["concepts"] = {
    "FA_lean": list(GECKO_PROMPTS["FA"].values()),
    "PT_lean": list(GECKO_PROMPTS["PT"].values()),
}

In [19]:
# create concept prior
import os
import glob
import argparse
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn.functional as F
from PIL import Image


# -------------------------
# NPY -> PIL
# -------------------------
def load_npy_as_pil(npy_path: str) -> Image.Image:
    arr = np.load(npy_path)
    if arr.dtype != np.uint8:
        arr = np.clip(arr, 0, 255).astype(np.uint8)
    if arr.ndim == 2:
        arr = np.stack([arr] * 3, axis=-1)
    if arr.ndim == 3 and arr.shape[-1] == 4:
        arr = arr[..., :3]
    return Image.fromarray(arr)


# -------------------------
# Text encoding (matches your working signature)
# -------------------------
def encode_text_prompts(prompts: List[str], tokenizer, model, device: str) -> np.ndarray:
    enc = tokenizer(prompts, padding=True, return_tensors="pt")
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.inference_mode():
        txt = model.encode_text(enc["input_ids"])
        txt = F.normalize(txt, dim=-1)
    return txt.detach().cpu().numpy()


# -------------------------
# Image encoding
# -------------------------
def encode_patch_embedding(npy_path: str, preprocess, model, device: str) -> np.ndarray:
    img = load_npy_as_pil(npy_path)
    x = preprocess(img).unsqueeze(0).to(device)
    with torch.inference_mode():
        e = model.encode_image(x)
        e = F.normalize(e, dim=-1)
    return e.detach().cpu().numpy()[0]


# -------------------------
# Prompt-bank builders
# -------------------------
def build_flat_class_prompts(P: dict) -> Tuple[List[str], np.ndarray, List[str]]:
    """
    templates × class synonyms -> flat prompts + class index map
    """
    templates = P["templates"]
    classes = P["classes"]
    class_names = list(classes.keys())  # ["FA","PT","NORMAL"] in your dict

    flat_prompts: List[str] = []
    flat_class_map: List[int] = []

    for ci, cls in enumerate(class_names):
        for syn in classes[cls]:
            for t in templates:
                flat_prompts.append(t.format(syn))
                flat_class_map.append(ci)

    return flat_prompts, np.array(flat_class_map, dtype=int), class_names


def build_flat_concepts(P: dict) -> Tuple[List[str], List[str], Dict[str, List[int]]]:
    """
    Flatten all concept prompts; also return indices per concept group.
    Dedupe while preserving order (important).
    """
    concept_list: List[str] = []
    concept_group: List[str] = []
    for grp, plist in P["concepts"].items():
        for p in plist:
            concept_list.append(p)
            concept_group.append(grp)

    # dedupe preserving order
    seen = set()
    out_list, out_group = [], []
    for p, g in zip(concept_list, concept_group):
        if p not in seen:
            out_list.append(p)
            out_group.append(g)
            seen.add(p)

    concept_list = out_list
    concept_group = out_group

    group_to_indices: Dict[str, List[int]] = {}
    for i, g in enumerate(concept_group):
        group_to_indices.setdefault(g, []).append(i)

    return concept_list, concept_group, group_to_indices


# -------------------------
# Patch scoring
# -------------------------
def softmax_np(x: np.ndarray) -> np.ndarray:
    x = x - np.max(x)
    e = np.exp(x)
    return e / (e.sum() + 1e-12)


def patch_class_probs(img_emb: np.ndarray, text_emb_class: np.ndarray, flat_class_map: np.ndarray, n_classes: int) -> np.ndarray:
    """
    For stability:
      sims -> softmax across ALL prompts -> average mass per class -> normalize
    """
    sims = img_emb @ text_emb_class.T  # (M,)
    p_prompt = softmax_np(sims)        # (M,)

    out = np.zeros(n_classes, dtype=float)
    for c in range(n_classes):
        out[c] = p_prompt[flat_class_map == c].mean()
    out = out / (out.sum() + 1e-12)
    return out  # (C,)


def patch_concept_sims(img_emb: np.ndarray, text_emb_concepts: np.ndarray) -> np.ndarray:
    """
    Raw cosine similarity per concept prompt
    """
    return img_emb @ text_emb_concepts.T  # (Nconcept,)

def load_slide_pt(pt_path: str):
    obj = torch.load(pt_path, map_location="cpu", weights_only=True)

    feats = obj["features"].float()
    coords = obj["coords"].int()

    feats = F.normalize(feats, dim=1)

    return feats.numpy(), coords.numpy()

# mass vis h5 feats---------------
import h5py
import torch
import torch.nn.functional as F

def load_slide_h5(h5_path: str):
    import h5py
    import torch
    import torch.nn.functional as F

    with h5py.File(h5_path, "r") as f:
        # embeddings may be nested inside a group
        if isinstance(f["embeddings"], h5py.Group):
            inner_key = list(f["embeddings"].keys())[0]
            feats = f["embeddings"][inner_key][:]
        else:
            feats = f["embeddings"][:]

        coords = f["embed_loc"][:]

    feats = torch.from_numpy(feats).float()
    feats = F.normalize(feats, dim=1)

    return feats.numpy(), coords.astype(int)
# --------------------------------

# -------------------------
# Slide scoring (aggregate patches)
# -------------------------
def classify_slide_pt(
    slide_id: str,
    pt_path: str,
    text_emb_class: np.ndarray,
    flat_class_map: np.ndarray,
    class_names: List[str],
    text_emb_concepts: np.ndarray,
    group_to_indices: Dict[str, List[int]],
    max_patches: Optional[int] = 800,
    topk_patches: Optional[int] = 200,
    select_mode: str = "conf",
    select_bucket: str = "PT_lean",
    seed: int = 0,
):
    # E, coords = load_slide_pt(pt_path) # og

    E, coords = load_slide_h5(pt_path) # h5 mass vis feats
    print(E.shape)       # should be (N, 512)
    print(coords.shape)  # should be (N, 2)

    N = E.shape[0]

    if max_patches is not None and N > max_patches:
        rng = np.random.default_rng(seed)
        idx_cap = rng.choice(N, size=max_patches, replace=False)
        E = E[idx_cap]
        coords = coords[idx_cap]

    # ---- vectorized scoring ----
    sims_class = E @ text_emb_class.T
    sims_class = sims_class - sims_class.max(axis=1, keepdims=True)
    p_prompt = np.exp(sims_class)
    p_prompt /= p_prompt.sum(axis=1, keepdims=True)

    n_classes = len(class_names)
    patch_probs = np.zeros((E.shape[0], n_classes))
    for c in range(n_classes):
        patch_probs[:, c] = p_prompt[:, flat_class_map == c].mean(axis=1)

    patch_probs /= patch_probs.sum(axis=1, keepdims=True)

    patch_conf = patch_probs.max(axis=1)

    patch_concept = E @ text_emb_concepts.T

    # selection
    if select_mode == "conf":
        select_score = patch_conf
    else:
        inds = group_to_indices[select_bucket]
        select_score = patch_concept[:, inds].mean(axis=1)

    if topk_patches is not None and topk_patches < E.shape[0]:
        idx = np.argsort(-select_score)[:topk_patches]
        patch_probs = patch_probs[idx]
        patch_concept = patch_concept[idx]
        coords = coords[idx]

    # slide_probs = patch_probs.mean(axis=0)
    # pred_idx = int(np.argmax(slide_probs))
    # pred_name = class_names[pred_idx]
    
    slide_probs = patch_probs.mean(axis=0)
    pred_idx = int(np.argmax(slide_probs))
    pred_name = class_names[pred_idx]
    margin = float(np.sort(slide_probs)[-1] - np.sort(slide_probs)[-2]) if len(slide_probs) >= 2 else np.nan


    mean_concept = patch_concept.mean(axis=0)
    # mean_concept = patch_concept.max(axis=0)

    topc_idx = np.argsort(-mean_concept)[:15]
    top_concepts = [(int(i), float(mean_concept[i])) for i in topc_idx]


    bucket_scores = {}
    for grp, inds in group_to_indices.items():
        bucket_scores[f"bucket_{grp}"] = float(np.mean(mean_concept[inds]))

    return {
    "slide_probs": slide_probs,
    "pred_name": pred_name,
    "margin": margin,
    "n_patches_scored": int(N),            # before cap/topk (or after cap; your choice)
    "n_patches_used": int(patch_probs.shape[0]),
    "bucket_scores": bucket_scores,
    "top_concepts": top_concepts,          # if you still want it
    "mean_concept": mean_concept,

    # NEW: patch-level outputs
    "patch_concept": patch_concept,   # (n_used, n_concepts)
    "patch_probs": patch_probs,       # (n_used, n_classes) optional
    "coords": coords,                 # (n_used, 2)
}



# -------------------------
# Dataset scanning
# -------------------------
# def list_slide_pt_files(feat_root: str):
#     pt_files = []
#     for cls in ["FA", "PT"]:
#         cls_root = os.path.join(feat_root, cls)
#         if not os.path.isdir(cls_root):
#             continue
#         for fname in os.listdir(cls_root):
#             if fname.endswith(".pt"):
#                 slide_id = os.path.splitext(fname)[0]
#                 pt_path = os.path.join(cls_root, fname)
#                 pt_files.append((slide_id, pt_path))
#     return pt_files

def list_slide_dirs(mag_root: str):
    out = []
    pt_files = sorted(glob.glob(os.path.join(mag_root, "*.pt")))
    for p in pt_files:
        slide_id = os.path.splitext(os.path.basename(p))[0]
        out.append((slide_id, p))
    return out



# def infer_label_from_path(slide_dir: str) -> str:
#     # label string from parent folder name
#     return os.path.basename(os.path.dirname(slide_dir)).upper()

def infer_label_from_path(pt_path: str) -> str:
    slide_id = os.path.splitext(os.path.basename(pt_path))[0]
    return slide_id.split()[0].upper()  # assumes "FA 56B", "PT 12B"

import os, glob, re

def list_slide_pt_files(mag_root: str):
    # mag_root example: ...\40x\pt
    files = sorted(glob.glob(os.path.join(mag_root, "*.pt")))
    return [(os.path.splitext(os.path.basename(p))[0], p) for p in files]

def infer_label_from_slide_id(slide_id: str) -> str:
    # handles "FA 47 B1", "FA 56B", "PT 12 B2", etc.
    m = re.match(r"^\s*(FA|PT)\b", slide_id.upper())
    if not m:
        raise ValueError(f"Cannot infer label from slide_id='{slide_id}'")
    return m.group(1)


# -------------------------
# MAIN
# -------------------------
def main_concept_prior():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--patch_root",
        type=str,
        default=r"C:\Users\Vivian\Documents\CONCH\all_patches\patches_2.5x",
        help="Root containing 20x/40x subdirs.",
    )
    parser.add_argument("--mag", type=str, default="40x", choices=["20x", "40x"])
    parser.add_argument(
        "--out_csv",
        type=str,
        default=r"C:\Users\Vivian\Documents\CONCH\test_text_encoder\2.5x_noNORM_conch_zeroshot_slides_fullprompts_40x.csv",
    )
    parser.add_argument(
        "--out_top_concepts_csv",
        type=str,
        default=r"C:\Users\Vivian\Documents\CONCH\test_text_encoder\2.5x_noNORM_conch_zeroshot_slides_top_concepts_40x.csv",
        help="Optional: per-slide top concepts (mean similarity) saved here.",
    )
    parser.add_argument("--max_patches", type=int, default=800)
    parser.add_argument("--topk_patches", type=int, default=200)
    parser.add_argument("--device", type=str, default="cuda" if torch.cuda.is_available() else "cpu")
    parser.add_argument("--seed", type=int, default=0)
    parser.add_argument("--select_mode", type=str, default="conf", choices=["conf", "concept"])
    parser.add_argument("--select_bucket", type=str, default="PT_lean")

    # args = parser.parse_args()
    args, _ = parser.parse_known_args()


    # expects these already exist in your environment
    global model, preprocess, tokenizer
    if "model" not in globals() or "preprocess" not in globals() or "tokenizer" not in globals():
        raise RuntimeError(
            "Expected `model`, `preprocess`, `tokenizer` to be defined already.\n"
            "Run this in your notebook, or paste your CONCH loading code above main()."
        )

    device = args.device
    model.to(device)
    model.eval()

    # mag_root = os.path.join(args.patch_root, args.mag, "pt")
    # og ---------------------
    # mag_root = os.path.join(args.patch_root)
    # if not os.path.isdir(mag_root):
    #     raise RuntimeError(f"Mag root not found: {mag_root}")
    # og ---------------------

    # Build + encode class prompts
    flat_class_prompts, flat_class_map, class_names = build_flat_class_prompts(PROMPTS)
    text_emb_class = encode_text_prompts(flat_class_prompts, tokenizer, model, device)

    # Build + encode concept prompts
    concept_list, concept_group, group_to_indices = build_flat_concepts(PROMPTS)
    text_emb_concepts = encode_text_prompts(concept_list, tokenizer, model, device)

    # Scan slide directories
    # slides = list_slide_dirs(mag_root)
    # slides = list_slide_pt_files(mag_root) # og pt files

    # h5 mass vis ----------------------
    def list_slide_h5_files(root):
        files = sorted(glob.glob(os.path.join(root, "*.h5")))
        return [(os.path.splitext(os.path.basename(p))[0], p) for p in files]

    # slides = list_slide_h5_files(mag_root)
    slides = [
    (
        "FA 113 B1",  # slide_id (must match label parser)
        r"C:\Users\Vivian\Documents\TRIDENT\slideio\FA 113 B1_224x224_10x_conch_emb.h5"
    )
]
    # -----------------------------------------

    # print(f"Found {len(slides)} slide folders under {mag_root}") # og 

    rows = []
    top_rows = []
    patch_rows = []

    # save concept index->text map once
    concept_map_path = os.path.splitext(args.out_csv)[0] + "_concept_map.csv"
    pd.DataFrame({
        "concept_idx": np.arange(len(concept_list)),
        "concept_group": concept_group,
        "concept_text": concept_list,
    }).to_csv(concept_map_path, index=False)
    print("Saved:", concept_map_path)


    for slide_id, pt_path in tqdm(slides, desc=f"Scoring slides ({args.mag})"):
        # true_label_str = infer_label_from_path(pt_path)  # "FA" or "PT" or "NORMAL"
        true_label_str = infer_label_from_slide_id(slide_id)

        res = classify_slide_pt(
        slide_id=slide_id,
        pt_path=pt_path,
        text_emb_class=text_emb_class,
        flat_class_map=flat_class_map,
        class_names=class_names,
        text_emb_concepts=text_emb_concepts,
        group_to_indices=group_to_indices,
        max_patches=args.max_patches if args.max_patches > 0 else None,
        topk_patches=args.topk_patches if args.topk_patches > 0 else None,
        select_mode=args.select_mode,
        select_bucket=args.select_bucket,
        seed=args.seed,
    )

        pc = res["patch_concept"]   # (n_used, K)
        cr = res["coords"]          # (n_used, 2)
        pp = res["patch_probs"]

        # for i in range(pc.shape[0]):
        #     r = {
        #         "mag": args.mag,
        #         "slide_id": slide_id,
        #         "true_label_str": true_label_str,
        #         "x": int(cr[i, 0]),
        #         "y": int(cr[i, 1]),
        #     }
        #     for j, cname in enumerate(concept_list):
        #         r[f"concept_{j}"] = float(pc[i, j])   # use index to keep columns sane
        #     patch_rows.append(r)
        
        # pp = res["patch_probs"]   # (n_used, C) optional but useful

        for i in range(pc.shape[0]):
            r = {
                "mag": args.mag,
                "slide_id": slide_id,
                "true_label_str": true_label_str,
                "x": int(cr[i, 0]),
                "y": int(cr[i, 1]),
                "conf": float(pp[i].max()),  # optional
            }
            # optional class probs
            for ci, cls in enumerate(class_names):
                r[f"prob_{cls.lower()}"] = float(pp[i, ci])

            for j in range(pc.shape[1]):
                r[f"concept_{j}"] = float(pc[i, j])
            patch_rows.append(r)


        
        slide_probs = res["slide_probs"]
        pred_name = res["pred_name"]

        row = {
            "mag": args.mag,
            "slide_id": slide_id,
            "true_label_str": true_label_str,
            "pred_class": pred_name,
            "margin": res["margin"],
            "n_patches_scored": res["n_patches_scored"],
            "n_patches_used": res["n_patches_used"],
            "pt_path": pt_path,
        }

        # add class probs
        for i, cls in enumerate(class_names):
            row[f"prob_{cls.lower()}"] = float(slide_probs[i])

        # add bucket scores
        row.update(res["bucket_scores"])

        # for i, cname in enumerate(concept_list):
        #     row[f"concept_{cname}"] = float(res["mean_concept"][i])

        for i in range(len(concept_list)):
            row[f"concept_{i}"] = float(res["mean_concept"][i])


        rows.append(row)

        # record top concepts
        for rank, (ci, score) in enumerate(res["top_concepts"], start=1):
            top_rows.append({
                "mag": args.mag,
                "slide_id": slide_id,
                "true_label_str": true_label_str,
                "pred_class": pred_name,
                "rank": rank,
                "concept_group": concept_group[ci],
                "concept": concept_list[ci],
                "mean_similarity": score,
            })

    patch_out = os.path.splitext(args.out_csv)[0] + "_PATCH.csv"
    pd.DataFrame(patch_rows).to_csv(patch_out, index=False)
    print("Saved:", patch_out)

    
    df = pd.DataFrame(rows)
    df.to_csv(args.out_csv, index=False)
    print("Saved:", args.out_csv)

    df_top = pd.DataFrame(top_rows)
    df_top.to_csv(args.out_top_concepts_csv, index=False)
    print("Saved:", args.out_top_concepts_csv)

    # Quick sanity summary (not CV-controlled)
    if len(df) > 0:
        # only compute "accuracy" where true_label_str is one of the model classes
        mask = df["true_label_str"].isin(class_names)
        if mask.any():
            acc = (df.loc[mask, "pred_class"].values == df.loc[mask, "true_label_str"].values).mean()
            print(f"\nNaive accuracy on this directory (not CV-controlled): {acc:.3f}")

        print("\nPrediction counts:")
        print(df["pred_class"].value_counts())

        print("\nMean bucket scores by predicted class:")
        bucket_cols = [c for c in df.columns if c.startswith("bucket_")]
        if bucket_cols:
            print(df.groupby("pred_class")[bucket_cols].mean())



In [20]:
import sys

sys.argv = [
    "notebook",
    "--patch_root", r"C:\dummy",   # not used anymore
    "--mag", "40x",
    "--out_csv", r"C:\Users\Vivian\Documents\CONCH\test_text_encoder\FA_113_B1_massVis_conf.csv",
    "--out_top_concepts_csv", r"C:\Users\Vivian\Documents\CONCH\test_text_encoder\FA_113_B1_massVis_topconcepts.csv",
    "--max_patches", "0",
    "--topk_patches", "0",
    "--select_mode", "conf",
    "--seed", "0",
]

main_concept_prior()

Saved: C:\Users\Vivian\Documents\CONCH\test_text_encoder\FA_113_B1_massVis_conf_concept_map.csv


Scoring slides (40x):   0%|          | 0/1 [00:00<?, ?it/s]

(16905, 512)
(16905, 2)


Scoring slides (40x): 100%|██████████| 1/1 [00:00<00:00,  3.36it/s]


Saved: C:\Users\Vivian\Documents\CONCH\test_text_encoder\FA_113_B1_massVis_conf_PATCH.csv
Saved: C:\Users\Vivian\Documents\CONCH\test_text_encoder\FA_113_B1_massVis_conf.csv
Saved: C:\Users\Vivian\Documents\CONCH\test_text_encoder\FA_113_B1_massVis_topconcepts.csv

Naive accuracy on this directory (not CV-controlled): 0.000

Prediction counts:
pred_class
PT    1
Name: count, dtype: int64

Mean bucket scores by predicted class:
            bucket_FA_lean  bucket_PT_lean  bucket_PT_benign_like  \
pred_class                                                          
PT                 0.14154        0.130493               0.122242   

            bucket_PT_borderline_like  bucket_PT_malignant_like  \
pred_class                                                        
PT                           0.138665                  0.067438   

            bucket_mitosis  bucket_NORMAL_lean  bucket_controls  
pred_class                                                       
PT                0.086585 

In [18]:
# load patch csv and check size
patch_Pt78_csv = r"C:\Users\Vivian\Documents\CONCH\test_text_encoder\PT_78_B2_massVis_conf_PATCH.csv"
df_patch78 = pd.read_csv(patch_Pt78_csv)

# size
print(df_patch78.shape)  # should be (n_patches_used, 4 + n_concepts)

(27510, 58)


In [12]:
import h5py

with h5py.File(r"C:\Users\Vivian\Documents\TRIDENT\slideio\PT 78 B2_224x224_10x_conch_emb.h5", "r") as f:
    print("Top level keys:", list(f.keys()))

Top level keys: ['embed_loc', 'embeddings', 'path_rect']


In [15]:
with h5py.File(r"C:\Users\Vivian\Documents\TRIDENT\slideio\PT 78 B2_224x224_10x_conch_emb.h5", "r") as f:
    print("Top:", list(f.keys()))
    print("embeddings contains:", list(f["embeddings"].keys()))

Top: ['embed_loc', 'embeddings', 'path_rect']
embeddings contains: ['conch_v1']


In [135]:
import sys

sys.argv = [
    "notebook",
    "--patch_root", r"C:\Users\Vivian\Documents\CONCH\conch_img_feats\10x_feats\conchextracted_mag10x_patch224_fp\feats_pt",  # <-- PT root
    "--mag", "40x",
    "--out_csv", r"C:\Users\Vivian\Documents\CONCH\test_text_encoder\slide_concept_scores\10x\GECKO_noCAP_patch_ptfile_notopk_conf.csv",
    "--out_top_concepts_csv", r"C:\Users\Vivian\Documents\CONCH\test_text_encoder\slide_concept_scores\10x\GECKO_noCAP_patch_ptfile_notopk_conf_topconcepts.csv",
    "--max_patches", "0",      # 0 = no cap
    "--topk_patches", "0",      # 0 = no top-k
    "--select_mode", "conf",    # or "concept"
    "--seed", "0",
]

main_concept_prior()

Found 240 slide folders under C:\Users\Vivian\Documents\CONCH\conch_img_feats\10x_feats\conchextracted_mag10x_patch224_fp\feats_pt
Saved: C:\Users\Vivian\Documents\CONCH\test_text_encoder\slide_concept_scores\10x\GECKO_noCAP_patch_ptfile_notopk_conf_concept_map.csv


Scoring slides (40x): 100%|██████████| 240/240 [00:17<00:00, 13.77it/s]


Saved: C:\Users\Vivian\Documents\CONCH\test_text_encoder\slide_concept_scores\10x\GECKO_noCAP_patch_ptfile_notopk_conf_PATCH.csv
Saved: C:\Users\Vivian\Documents\CONCH\test_text_encoder\slide_concept_scores\10x\GECKO_noCAP_patch_ptfile_notopk_conf.csv
Saved: C:\Users\Vivian\Documents\CONCH\test_text_encoder\slide_concept_scores\10x\GECKO_noCAP_patch_ptfile_notopk_conf_topconcepts.csv

Naive accuracy on this directory (not CV-controlled): 0.725

Prediction counts:
pred_class
FA    141
PT     99
Name: count, dtype: int64

Mean bucket scores by predicted class:
            bucket_FA_lean  bucket_PT_lean
pred_class                                
FA                0.260040        0.144539
PT                0.293472        0.201890


In [100]:
import os
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, accuracy_score, confusion_matrix

# ----------------------------
# Config
# ----------------------------
PATCH_CSV = r"C:\Users\Vivian\Documents\CONCH\test_text_encoder\slide_concept_scores\noCAP_patch_ptfile_notopk_conf_PATCH.csv"          # patch-level CSV you saved
CONCEPT_MAP = r"C:\Users\Vivian\Documents\CONCH\test_text_encoder\slide_concept_scores\noCAP_patch_ptfile_notopk_conf_concept_map.csv" # idx->text/group map (optional)
CV_ROOT = r"C:/Users/Vivian/Documents/PANTHER/PANTHER/src/splits/cross-val"                     # contains fold dirs with train/val/test.csv
USE_TRAIN_PLUS_VAL = True
SEED = 0

SLIDE_COL = "slide_id"
LABEL_COL = "true_label_str"   # "FA"/"PT"
CONCEPT_PREFIX = "concept_"

# ----------------------------
# Helpers
# ----------------------------
def labels_to_int(y_str: pd.Series) -> np.ndarray:
    y = y_str.astype(str).str.upper().str.strip().map({"FA": 0, "PT": 1})
    if y.isna().any():
        bad = y_str[y.isna()].unique()
        raise ValueError(f"Unexpected labels in {LABEL_COL}: {bad} (expected FA/PT)")
    return y.astype(int).to_numpy()

def read_slide_list(csv_path: str):
    d = pd.read_csv(csv_path)
    if "slide" in d.columns:
        slides = d["slide"].astype(str).tolist()
    elif "slide_id" in d.columns:
        slides = d["slide_id"].astype(str).tolist()
    else:
        slides = d.iloc[:, 0].astype(str).tolist()
    # normalize: drop extension + strip
    return [os.path.splitext(s.strip())[0] for s in slides]

def find_fold_dirs(cv_root: str):
    # works whether folds are named "fold_0" or "FA_PT_k=..." etc.
    out = []
    for name in sorted(os.listdir(cv_root)):
        p = os.path.join(cv_root, name)
        if os.path.isdir(p) and all(os.path.isfile(os.path.join(p, f"{x}.csv")) for x in ["train","val","test"]):
            out.append(p)
    return out

def slide_level_from_patch_df(df_patch: pd.DataFrame):
    concept_cols = [c for c in df_patch.columns if c.startswith(CONCEPT_PREFIX)]
    if not concept_cols:
        raise ValueError("No concept_* columns found in patch CSV")

    # ensure numeric
    df_patch = df_patch.copy()
    for c in concept_cols:
        df_patch[c] = pd.to_numeric(df_patch[c], errors="coerce")

    # one row per slide: mean over patches (simple)
    agg = {c: "mean" for c in concept_cols}
    agg[LABEL_COL] = "first"
    if "mag" in df_patch.columns:
        agg["mag"] = "first"

    df_slide = df_patch.groupby(SLIDE_COL, as_index=False).agg(agg)

    # clean
    df_slide[concept_cols] = df_slide[concept_cols].replace([np.inf, -np.inf], np.nan)

    return df_slide, concept_cols

# def slide_level_from_patch_df(df_patch: pd.DataFrame, mode="mean", topk=None, conf_col="conf"):
#     concept_cols = [c for c in df_patch.columns if c.startswith("concept_")]
#     if not concept_cols:
#         raise ValueError("No concept_* columns found")

#     df = df_patch.copy()
#     for c in concept_cols:
#         df[c] = pd.to_numeric(df[c], errors="coerce")

#     # optional: top-k patches per slide by confidence
#     if topk is not None:
#         if conf_col not in df.columns:
#             raise ValueError(f"topk requested but '{conf_col}' not in patch CSV")
#         df = df.sort_values([SLIDE_COL, conf_col], ascending=[True, False])
#         df = df.groupby(SLIDE_COL, as_index=False).head(topk)

#     agg = {LABEL_COL: "first"}
#     if "mag" in df.columns:
#         agg["mag"] = "first"

#     if mode == "mean":
#         for c in concept_cols: agg[c] = "mean"
#     elif mode == "max":
#         for c in concept_cols: agg[c] = "max"
#     else:
#         raise ValueError("mode must be 'mean' or 'max'")

#     df_slide = df.groupby(SLIDE_COL, as_index=False).agg(agg)
#     df_slide[concept_cols] = df_slide[concept_cols].replace([np.inf, -np.inf], np.nan)
#     return df_slide, concept_cols


def eval_prob(y_true, prob, thr=0.5):
    pred = (prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {
        "auc": roc_auc_score(y_true, prob) if len(np.unique(y_true)) == 2 else np.nan,
        "balacc": balanced_accuracy_score(y_true, pred),
        "acc": accuracy_score(y_true, pred),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

# ----------------------------
# Load + aggregate patch -> slide
# ----------------------------
df_patch = pd.read_csv(PATCH_CSV)
df_slide, concept_cols = slide_level_from_patch_df(df_patch)
df_slide[SLIDE_COL] = df_slide[SLIDE_COL].astype(str).str.strip()

df_slide = df_slide.set_index(SLIDE_COL, drop=False)
print("Slides in patch CSV:", len(df_slide), " | #concept features:", len(concept_cols))

# testing
# df_patch = pd.read_csv(PATCH_CSV)

# for name, (mode, topk) in {
#     "mean_all": ("mean", None),
#     "max_all":  ("max",  None),
#     "mean_top50": ("mean", 50),
# }.items():
#     df_slide, concept_cols = slide_level_from_patch_df(df_patch, mode=mode, topk=topk)

#     # run your same 5-fold LR loop using df_slide + concept_cols
#     # (everything else unchanged)
#     print("\n===", name, "=== slides:", len(df_slide))


# ----------------------------
# CV
# ----------------------------
fold_dirs = find_fold_dirs(CV_ROOT)
if not fold_dirs:
    raise ValueError(f"No fold dirs found under: {CV_ROOT}")

fold_rows = []
all_test_preds = []

pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=True, with_std=True)),
    ("clf", LogisticRegression(
        solver="liblinear",
        class_weight="balanced",
        random_state=SEED,
        max_iter=5000,
    )),
])

for fold_dir in fold_dirs:
    fold_name = os.path.basename(fold_dir)

    train_slides = read_slide_list(os.path.join(fold_dir, "train.csv"))
    val_slides   = read_slide_list(os.path.join(fold_dir, "val.csv"))
    test_slides  = read_slide_list(os.path.join(fold_dir, "test.csv"))

    fit_slides = train_slides + (val_slides if USE_TRAIN_PLUS_VAL else [])
    fit_slides = list(dict.fromkeys(fit_slides))  # unique preserve order

    fit_in  = [s for s in fit_slides if s in df_slide.index]
    test_in = [s for s in test_slides if s in df_slide.index]

    if len(test_in) == 0:
        print(f"[{fold_name}] WARNING: no test slides matched; skipping")
        continue

    Xtr = df_slide.loc[fit_in, concept_cols]
    ytr = labels_to_int(df_slide.loc[fit_in, LABEL_COL])

    Xte = df_slide.loc[test_in, concept_cols]
    yte = labels_to_int(df_slide.loc[test_in, LABEL_COL])

    pipe.fit(Xtr, ytr)
    prob = pipe.predict_proba(Xte)[:, 1]

    m = eval_prob(yte, prob, thr=0.5)
    fold_rows.append({
        "fold": fold_name,
        "n_train": len(fit_in),
        "n_test": len(test_in),
        **m,
    })

    all_test_preds.append(pd.DataFrame({
        "fold": fold_name,
        "slide_id": test_in,
        "true_label": df_slide.loc[test_in, LABEL_COL].values,
        "prob_pt": prob,
        "pred_label": np.where(prob >= 0.5, "PT", "FA"),
    }))

    print(f"[{fold_name}] auc={m['auc']:.3f} balacc={m['balacc']:.3f} acc={m['acc']:.3f} (n_test={len(test_in)})")

results = pd.DataFrame(fold_rows)
print("\n=== 5-fold summary (mean ± std) ===")
print(results[["auc","balacc","acc"]].agg(["mean","std"]))

# optional: save preds + metrics next to PATCH_CSV
# out_dir = os.path.dirname(PATCH_CSV)
# results.to_csv(os.path.join(out_dir, "cv5_patchConcept_slideMax_metrics.csv"), index=False)
# pd.concat(all_test_preds, ignore_index=True).to_csv(os.path.join(out_dir, "cv5_patchConcept_slideMax_testpreds.csv"), index=False)
# print("\nSaved metrics/preds to:", out_dir)


Slides in patch CSV: 240  | #concept features: 50


Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=5000,
                                    random_state=0, solver='liblinear'))])

[FA_PT_k=0] auc=0.780 balacc=0.664 acc=0.660 (n_test=47)


Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=5000,
                                    random_state=0, solver='liblinear'))])

[FA_PT_k=1] auc=0.812 balacc=0.706 acc=0.708 (n_test=48)


...

[FA_PT_k=2] auc=0.868 balacc=0.780 acc=0.778 (n_test=45)


Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=5000,
                                    random_state=0, solver='liblinear'))])

[FA_PT_k=3] auc=0.875 balacc=0.753 acc=0.755 (n_test=49)


Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=5000,
                                    random_state=0, solver='liblinear'))])

[FA_PT_k=4] auc=0.729 balacc=0.685 acc=0.686 (n_test=51)

=== 5-fold summary (mean ± std) ===
           auc    balacc       acc
mean  0.812881  0.717688  0.717412
std   0.061188  0.048096  0.048636


In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, accuracy_score
import h5py


PATCH_CSV = r"C:\Users\Vivian\Documents\CONCH\test_text_encoder\slide_concept_scores\noCAP_patch_ptfile_notopk_conf_PATCH.csv"
CV_ROOT   = r"C:/Users/Vivian/Documents/PANTHER/PANTHER/src/splits/cross-val"
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
SEED      = 0

# for deep feats
# PT_FEAT_DIR = r"C:\Users\Vivian\Documents\CONCH\conch_img_feats\10x_feats\40x\pt"  # folder with *.pt
FEAT_KEY = "features"   # your .pt dict has keys: features, coords
COORD_KEY = "coords"

# uni h5 
H5_FEAT_DIR = r"C:\Users\Vivian\Documents\CLAM\CLAM\FEATURES_DIR_5x\FEATURES_DIR_10x\uniextracted_mag10x_patch224_fp\feats_h5"


def load_slide_pt(pt_path: str):
    obj = torch.load(pt_path, map_location="cpu", weights_only=True)
    feats = obj[FEAT_KEY].float()          # (N, D)
    coords = obj[COORD_KEY].int()          # (N, 2) optional
    feats = torch.nn.functional.normalize(feats, dim=1)  # IMPORTANT (cosine-like)
    return feats.numpy(), coords.numpy()

def load_slide_h5(h5_path: str):
    with h5py.File(h5_path, "r") as f:
        # common CLAM keys
        if "features" in f:
            feats = f["features"][:]
        elif "feats" in f:
            feats = f["feats"][:]
        else:
            raise KeyError(f"No 'features' or 'feats' in {h5_path}. Keys={list(f.keys())}")

        if "coords" in f:
            coords = f["coords"][:]
        else:
            raise KeyError(f"No 'coords' in {h5_path}. Needed for gating/alignment. Keys={list(f.keys())}")

    return feats.astype(np.float32), coords.astype(np.int32)

# ---------------

SLIDE_COL = "slide_id"
LABEL_COL = "true_label_str"
CONCEPT_PREFIX = "concept_"

torch.manual_seed(SEED)
np.random.seed(SEED)

def labels_to_int(arr):
    s = pd.Series(arr).astype(str).str.upper().str.strip()
    y = s.map({"FA": 0, "PT": 1})
    if y.isna().any():
        raise ValueError(f"Bad labels: {s[y.isna()].unique()}")
    return y.astype(int).to_numpy()

def read_slide_list(csv_path: str):
    d = pd.read_csv(csv_path)
    if "slide" in d.columns:
        slides = d["slide"].astype(str).tolist()
    elif "slide_id" in d.columns:
        slides = d["slide_id"].astype(str).tolist()
    else:
        slides = d.iloc[:, 0].astype(str).tolist()
    return [os.path.splitext(s.strip())[0] for s in slides]

def find_fold_dirs(cv_root: str):
    out = []
    for name in sorted(os.listdir(cv_root)):
        p = os.path.join(cv_root, name)
        if os.path.isdir(p) and all(os.path.isfile(os.path.join(p, f"{x}.csv")) for x in ["train","val","test"]):
            out.append(p)
    return out

class BagDataset(Dataset):
    """
    Returns one bag (slide) at a time:
      X: (Npatch, K) float32
      y: scalar int64
    """
    def __init__(self, df_patch, slide_ids, concept_cols, max_patches=None, use_conf_topk=None):
        self.df = df_patch[df_patch[SLIDE_COL].isin(slide_ids)].copy()
        self.slide_ids = [s for s in slide_ids if s in set(self.df[SLIDE_COL].unique())]
        self.concept_cols = concept_cols
        self.max_patches = max_patches
        self.use_conf_topk = use_conf_topk  # if not None, uses top-k by 'conf' column

        # label per slide
        lab = self.df.groupby(SLIDE_COL)[LABEL_COL].first()
        self.y_map = {sid: int(labels_to_int([lab.loc[sid]])[0]) for sid in self.slide_ids}

    def __len__(self):
        return len(self.slide_ids)

    def __getitem__(self, idx):
        sid = self.slide_ids[idx]
        d = self.df[self.df[SLIDE_COL] == sid]

        # optional: pick top-k patches by conf
        if self.use_conf_topk is not None:
            if "conf" not in d.columns:
                raise ValueError("use_conf_topk set but 'conf' column missing in patch CSV")
            d = d.sort_values("conf", ascending=False).head(self.use_conf_topk)

        X = d[self.concept_cols].to_numpy(dtype=np.float32)

        # optional: random cap
        if self.max_patches is not None and X.shape[0] > self.max_patches:
            ridx = np.random.choice(X.shape[0], size=self.max_patches, replace=False)
            X = X[ridx]

        y = self.y_map[sid]
        return torch.from_numpy(X), torch.tensor(y, dtype=torch.long), sid

def collate_bag(batch):
    # batch is list of (X, y, sid); variable length bags
    return batch  # keep as list

# deep feats =============
import re

def infer_label_from_slide_id(slide_id: str) -> int:
    m = re.match(r"^\s*(FA|PT)\b", slide_id.upper())
    if not m:
        raise ValueError(f"Cannot infer label from slide_id='{slide_id}'")
    return 0 if m.group(1) == "FA" else 1

class BagDatasetDeep(Dataset):
    """
    Loads one slide bag from a .pt file:
      X: (Npatch, D) float32
      y: scalar int64
    """
    def __init__(self, slide_ids, pt_dir, max_patches=None):
        self.slide_ids = slide_ids
        self.pt_dir = pt_dir
        self.max_patches = max_patches

        # keep only slides that exist as pt
        keep = []
        for sid in self.slide_ids:
            pt_path = os.path.join(self.pt_dir, f"{sid}.pt")
            if os.path.isfile(pt_path):
                keep.append(sid)
        self.slide_ids = keep

        # labels from slide_id prefix (FA/PT)
        self.y_map = {sid: infer_label_from_slide_id(sid) for sid in self.slide_ids}

    def __len__(self):
        return len(self.slide_ids)

    def __getitem__(self, idx):
        # sid = self.slide_ids[idx]
        # pt_path = os.path.join(self.pt_dir, f"{sid}.pt")

        # X, coords = load_slide_pt(pt_path)   # X: (N, D)
        

        # if self.max_patches is not None and X.shape[0] > self.max_patches:
        #     ridx = np.random.choice(X.shape[0], size=self.max_patches, replace=False)
        #     X = X[ridx]

        # y = self.y_map[sid]
        # return torch.from_numpy(X.astype(np.float32)), torch.tensor(y, dtype=torch.long), sid

        # uni feats h5
        sid = self.slide_ids[idx]

        h5_path = os.path.join(self.pt_dir, f"{sid}.h5")
        if not os.path.isfile(h5_path):
            raise FileNotFoundError(h5_path)

        X, coords = load_slide_h5(h5_path)   # X:(N,D), coords:(N,2)

        # optional random cap (apply to both X and coords)
        if self.max_patches is not None and X.shape[0] > self.max_patches:
            ridx = np.random.choice(X.shape[0], size=self.max_patches, replace=False)
            X = X[ridx]
            coords = coords[ridx]

        y = self.y_map[sid]

        return (
            torch.from_numpy(X.astype(np.float32)),
            torch.from_numpy(coords.astype(np.int32)),
            torch.tensor(y, dtype=torch.long),
            sid
        )
# -----------------------------

class AttentionMIL(nn.Module):
    def __init__(self, in_dim, hid=128):
        super().__init__()
        self.phi = nn.Sequential(
            nn.Linear(in_dim, hid),
            nn.ReLU(),
            nn.Dropout(0.1),
        )
        self.attn = nn.Sequential(
            nn.Linear(hid, 1)
        )
        self.cls = nn.Linear(hid, 1)

    def forward(self, X):  # X: (N, K)
        H = self.phi(X)                 # (N, hid)
        a = self.attn(H).squeeze(1)     # (N,)
        w = torch.softmax(a, dim=0)     # (N,)
        z = (w.unsqueeze(1) * H).sum(0) # (hid,)
        logit = self.cls(z).squeeze(0)  # ()
        return logit, w

def eval_metrics(y_true, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(int)
    out = {
        "auc": roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else np.nan,
        "balacc": balanced_accuracy_score(y_true, y_pred),
        "acc": accuracy_score(y_true, y_pred),
    }
    return out

# ----------------------------
# Load patch CSV
# ----------------------------
df_patch = pd.read_csv(PATCH_CSV)
concept_cols = [c for c in df_patch.columns if c.startswith(CONCEPT_PREFIX)]
if not concept_cols:
    raise ValueError("No concept_* columns found in PATCH_CSV")

# numeric + impute NaNs to 0 (simple; you can do median too)
df_patch[concept_cols] = df_patch[concept_cols].apply(pd.to_numeric, errors="coerce")
df_patch[concept_cols] = df_patch[concept_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)

fold_dirs = find_fold_dirs(CV_ROOT)

# ----------------------------
# Run 5-fold MIL
# ----------------------------
all_fold = []

EPOCHS = 8
LR = 1e-3
WEIGHT_DECAY = 1e-4

# (optional) make bags smaller for speed:
BAG_CAP = None       # e.g., 500
CONF_TOPK = None     # e.g., 200 if you have 'conf' col

for fold_dir in fold_dirs:
    fold_name = os.path.basename(fold_dir)

    train_slides = read_slide_list(os.path.join(fold_dir, "train.csv"))
    val_slides   = read_slide_list(os.path.join(fold_dir, "val.csv"))
    test_slides  = read_slide_list(os.path.join(fold_dir, "test.csv"))

    fit_slides = train_slides + val_slides  # keep it simple

    ds_tr = BagDataset(df_patch, fit_slides, concept_cols, max_patches=BAG_CAP, use_conf_topk=CONF_TOPK)
    ds_te = BagDataset(df_patch, test_slides, concept_cols, max_patches=BAG_CAP, use_conf_topk=CONF_TOPK)

    # deep feats ---------
    # ds_tr = BagDatasetDeep(fit_slides, PT_FEAT_DIR, max_patches=BAG_CAP)
    # ds_te = BagDatasetDeep(test_slides, PT_FEAT_DIR, max_patches=BAG_CAP)
    # ------------------


    dl_tr = DataLoader(ds_tr, batch_size=1, shuffle=True, collate_fn=collate_bag)
    dl_te = DataLoader(ds_te, batch_size=1, shuffle=False, collate_fn=collate_bag)

    # deep feats ---------
    # X0, _, _ = ds_tr[0]
    # in_dim = X0.shape[1]

    # model = AttentionMIL(in_dim=in_dim, hid=128).to(DEVICE)
    # deep feat -------------------

    model = AttentionMIL(in_dim=len(concept_cols), hid=128).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.BCEWithLogitsLoss()

    # train
    model.train()
    for ep in range(EPOCHS):
        for batch in dl_tr:
            (X, y, sid) = batch[0]
            X = X.to(DEVICE)
            y = y.float().to(DEVICE)

            logit, _ = model(X)
            loss = loss_fn(logit.view(1), y.view(1))

            opt.zero_grad()
            loss.backward()
            opt.step()

    # eval
    model.eval()
    y_true, y_prob = [], []
    with torch.no_grad():
        for batch in dl_te:
            (X, y, sid) = batch[0]
            X = X.to(DEVICE)
            logit, w = model(X)
            prob = torch.sigmoid(logit).item()
            y_true.append(int(y.item()))
            y_prob.append(prob)

    y_true = np.array(y_true)
    y_prob = np.array(y_prob)

    m = eval_metrics(y_true, y_prob, thr=0.5)
    all_fold.append({"fold": fold_name, **m, "n_test": len(y_true)})
    print(f"[{fold_name}] auc={m['auc']:.3f} balacc={m['balacc']:.3f} acc={m['acc']:.3f} n_test={len(y_true)}")

res = pd.DataFrame(all_fold)
print("\n=== MIL 5-fold summary (mean ± std) ===")
print(res[["auc","balacc","acc"]].agg(["mean","std"]))


AttentionMIL(
  (phi): Sequential(
    (0): Linear(in_features=50, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (attn): Sequential(
    (0): Linear(in_features=128, out_features=1, bias=True)
  )
  (cls): Linear(in_features=128, out_features=1, bias=True)
)

AttentionMIL(
  (phi): Sequential(
    (0): Linear(in_features=50, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (attn): Sequential(
    (0): Linear(in_features=128, out_features=1, bias=True)
  )
  (cls): Linear(in_features=128, out_features=1, bias=True)
)

[FA_PT_k=0] auc=0.822 balacc=0.766 acc=0.766 n_test=47


AttentionMIL(
  (phi): Sequential(
    (0): Linear(in_features=50, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (attn): Sequential(
    (0): Linear(in_features=128, out_features=1, bias=True)
  )
  (cls): Linear(in_features=128, out_features=1, bias=True)
)

AttentionMIL(
  (phi): Sequential(
    (0): Linear(in_features=50, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (attn): Sequential(
    (0): Linear(in_features=128, out_features=1, bias=True)
  )
  (cls): Linear(in_features=128, out_features=1, bias=True)
)

...

AttentionMIL(
  (phi): Sequential(
    (0): Linear(in_features=50, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (attn): Sequential(
    (0): Linear(in_features=128, out_features=1, bias=True)
  )
  (cls): Linear(in_features=128, out_features=1, bias=True)
)

[FA_PT_k=3] auc=0.833 balacc=0.670 acc=0.673 n_test=49


AttentionMIL(
  (phi): Sequential(
    (0): Linear(in_features=50, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (attn): Sequential(
    (0): Linear(in_features=128, out_features=1, bias=True)
  )
  (cls): Linear(in_features=128, out_features=1, bias=True)
)

AttentionMIL(
  (phi): Sequential(
    (0): Linear(in_features=50, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (attn): Sequential(
    (0): Linear(in_features=128, out_features=1, bias=True)
  )
  (cls): Linear(in_features=128, out_features=1, bias=True)
)

[FA_PT_k=4] auc=0.698 balacc=0.645 acc=0.647 n_test=51

=== MIL 5-fold summary (mean ± std) ===
           auc    balacc       acc
mean  0.803592  0.711674  0.714242
std   0.059215  0.053835  0.051913


fuse concepts and deep feats

In [ ]:
# =============================
# PATCH-LEVEL CONCAT MIL (deep || concepts)
# minimal mod of your script
# =============================
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, accuracy_score

PATCH_CONCEPT_CSV = r"C:\Users\Vivian\Documents\CONCH\test_text_encoder\slide_concept_scores\noCAP_patch_ptfile_notopk_conf_PATCH.csv"
PT_FEAT_DIR       = r"C:\Users\Vivian\Documents\CONCH\conch_img_feats\10x_feats\40x\pt"
CV_ROOT           = r"C:/Users/Vivian/Documents/PANTHER/PANTHER/src/splits/cross-val"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)

SLIDE_COL = "slide_id"
LABEL_COL = "true_label_str"
CONCEPT_PREFIX = "concept_"

# -----------------------------
# Helpers
# -----------------------------
def labels_to_int(arr):
    s = pd.Series(arr).astype(str).str.upper().str.strip()
    y = s.map({"FA": 0, "PT": 1})
    if y.isna().any():
        raise ValueError(f"Bad labels: {s[y.isna()].unique()}")
    return y.astype(int).to_numpy()

def read_slide_list(csv_path: str):
    d = pd.read_csv(csv_path)
    if "slide" in d.columns:
        slides = d["slide"].astype(str).tolist()
    elif "slide_id" in d.columns:
        slides = d["slide_id"].astype(str).tolist()
    else:
        slides = d.iloc[:, 0].astype(str).tolist()
    return [os.path.splitext(s.strip())[0] for s in slides]

def find_fold_dirs(cv_root: str):
    out = []
    for name in sorted(os.listdir(cv_root)):
        p = os.path.join(cv_root, name)
        if os.path.isdir(p) and all(os.path.isfile(os.path.join(p, f"{x}.csv")) for x in ["train","val","test"]):
            out.append(p)
    return out

def load_slide_pt(pt_path: str):
    obj = torch.load(pt_path, map_location="cpu", weights_only=True)
    feats = obj["features"].float()  # (N,D)
    coords = obj["coords"].int()     # (N,2)
    feats = torch.nn.functional.normalize(feats, dim=1)
    return feats.numpy(), coords.numpy()

# -----------------------------
# Dataset: aligns concept rows to deep feats via coords
# -----------------------------
class ConcatBagDataset(Dataset):
    """
    One bag per slide.
    For each slide:
      - load deep feats (E, coords) from .pt
      - join with concept feats using (x,y) coords in PATCH_CONCEPT_CSV
      - return X_concat: (N, Ddeep + Kconcept)
    """
    def __init__(self, df_patch_concept, slide_ids, concept_cols,
                 pt_dir, max_patches=None, topk_patches=None, seed=0):
        self.df = df_patch_concept[df_patch_concept[SLIDE_COL].isin(slide_ids)].copy()
        self.slide_ids = [s for s in slide_ids if s in set(self.df[SLIDE_COL].unique())]
        self.concept_cols = concept_cols
        self.pt_dir = pt_dir
        self.max_patches = max_patches
        self.topk_patches = topk_patches
        self.rng = np.random.default_rng(seed)

        lab = self.df.groupby(SLIDE_COL)[LABEL_COL].first()
        self.y_map = {sid: int(labels_to_int([lab.loc[sid]])[0]) for sid in self.slide_ids}

        # pre-index concept df by slide for speed
        self.by_slide = {sid: d for sid, d in self.df.groupby(SLIDE_COL)}

    def __len__(self):
        return len(self.slide_ids)

    def __getitem__(self, idx):
        sid = self.slide_ids[idx]
        pt_path = os.path.join(self.pt_dir, f"{sid}.pt")
        if not os.path.isfile(pt_path):
            raise FileNotFoundError(pt_path)

        E, coords = load_slide_pt(pt_path)      # E: (N,D), coords: (N,2)
        d = self.by_slide[sid]

        # build map from (x,y)->concept vector
        # assumes PATCH csv uses columns named x,y (as you saved)
        key = list(zip(d["x"].astype(int).to_numpy(), d["y"].astype(int).to_numpy()))
        C = d[self.concept_cols].to_numpy(dtype=np.float32)
        concept_map = {k: C[i] for i, k in enumerate(key)}

        # align concepts to deep by coords
        Xc = np.zeros((coords.shape[0], len(self.concept_cols)), dtype=np.float32)
        miss = 0
        for i, (x, y) in enumerate(coords.astype(int)):
            v = concept_map.get((x, y), None)
            if v is None:
                miss += 1
            else:
                Xc[i] = v

        # concat
        X = np.concatenate([E.astype(np.float32), Xc], axis=1)

        # optional: random cap
        if self.max_patches is not None and X.shape[0] > self.max_patches:
            sel = self.rng.choice(X.shape[0], size=self.max_patches, replace=False)
            X = X[sel]

        # optional: top-k by L2 norm of deep part (no conf needed)
        if self.topk_patches is not None and X.shape[0] > self.topk_patches:
            deep_norm = np.linalg.norm(X[:, :E.shape[1]], axis=1)
            sel = np.argsort(-deep_norm)[:self.topk_patches]
            X = X[sel]

        y = self.y_map[sid]
        return torch.from_numpy(X), torch.tensor(y, dtype=torch.long), sid

def collate_bag(batch):
    return batch

# -----------------------------
# MIL model (same)
# -----------------------------
class AttentionMIL(nn.Module):
    def __init__(self, in_dim, hid=128):
        super().__init__()
        self.phi = nn.Sequential(
            nn.Linear(in_dim, hid),
            nn.ReLU(),
            nn.Dropout(0.1),
        )
        self.attn = nn.Linear(hid, 1)
        self.cls = nn.Linear(hid, 1)

    def forward(self, X):
        H = self.phi(X)
        a = self.attn(H).squeeze(1)
        w = torch.softmax(a, dim=0)
        z = (w.unsqueeze(1) * H).sum(0)
        logit = self.cls(z).squeeze(0)
        return logit, w

def eval_metrics(y_true, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(int)
    return {
        "auc": roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else np.nan,
        "balacc": balanced_accuracy_score(y_true, y_pred),
        "acc": accuracy_score(y_true, y_pred),
    }

# -----------------------------
# Load concept patch CSV
# -----------------------------
df_patch = pd.read_csv(PATCH_CONCEPT_CSV)
concept_cols = [c for c in df_patch.columns if c.startswith(CONCEPT_PREFIX)]
if not concept_cols:
    raise ValueError("No concept_* columns found in PATCH_CONCEPT_CSV")

# numeric + impute NaNs -> 0
df_patch[concept_cols] = df_patch[concept_cols].apply(pd.to_numeric, errors="coerce")
df_patch[concept_cols] = df_patch[concept_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)

fold_dirs = find_fold_dirs(CV_ROOT)

# -----------------------------
# Run 5-fold
# -----------------------------
EPOCHS = 8
LR = 1e-3
WEIGHT_DECAY = 1e-4

BAG_CAP = 800       # match your previous cap
TOPK = 200         # or 200, etc.

all_fold = []

for fold_dir in fold_dirs:
    fold_name = os.path.basename(fold_dir)

    train_slides = read_slide_list(os.path.join(fold_dir, "train.csv"))
    val_slides   = read_slide_list(os.path.join(fold_dir, "val.csv"))
    test_slides  = read_slide_list(os.path.join(fold_dir, "test.csv"))
    fit_slides = train_slides + val_slides

    ds_tr = ConcatBagDataset(df_patch, fit_slides, concept_cols, PT_FEAT_DIR,
                             max_patches=BAG_CAP, topk_patches=TOPK, seed=SEED)
    ds_te = ConcatBagDataset(df_patch, test_slides, concept_cols, PT_FEAT_DIR,
                             max_patches=BAG_CAP, topk_patches=TOPK, seed=SEED)

    dl_tr = DataLoader(ds_tr, batch_size=1, shuffle=True, collate_fn=collate_bag)
    dl_te = DataLoader(ds_te, batch_size=1, shuffle=False, collate_fn=collate_bag)

    # infer in_dim from first bag
    X0, _, _ = ds_tr[0]
    model = AttentionMIL(in_dim=X0.shape[1], hid=128).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.BCEWithLogitsLoss()

    # train
    model.train()
    for ep in range(EPOCHS):
        for batch in dl_tr:
            (X, y, sid) = batch[0]
            X = X.to(DEVICE)
            y = y.float().to(DEVICE)

            logit, _ = model(X)
            loss = loss_fn(logit.view(1), y.view(1))

            opt.zero_grad()
            loss.backward()
            opt.step()

    # eval
    model.eval()
    y_true, y_prob = [], []
    with torch.no_grad():
        for batch in dl_te:
            (X, y, sid) = batch[0]
            X = X.to(DEVICE)
            logit, w = model(X)
            prob = torch.sigmoid(logit).item()
            y_true.append(int(y.item()))
            y_prob.append(prob)

    y_true = np.array(y_true)
    y_prob = np.array(y_prob)

    m = eval_metrics(y_true, y_prob, thr=0.5)
    all_fold.append({"fold": fold_name, **m, "n_test": len(y_true)})
    print(f"[{fold_name}] auc={m['auc']:.3f} balacc={m['balacc']:.3f} acc={m['acc']:.3f} n_test={len(y_true)}")

res = pd.DataFrame(all_fold)
print("\n=== CONCAT MIL (deep||concept) 5-fold summary (mean ± std) ===")
print(res[["auc","balacc","acc"]].agg(["mean","std"]))


AttentionMIL(
  (phi): Sequential(
    (0): Linear(in_features=562, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (attn): Linear(in_features=128, out_features=1, bias=True)
  (cls): Linear(in_features=128, out_features=1, bias=True)
)

AttentionMIL(
  (phi): Sequential(
    (0): Linear(in_features=562, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (attn): Linear(in_features=128, out_features=1, bias=True)
  (cls): Linear(in_features=128, out_features=1, bias=True)
)

[FA_PT_k=0] auc=0.816 balacc=0.706 acc=0.702 n_test=47


AttentionMIL(
  (phi): Sequential(
    (0): Linear(in_features=562, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (attn): Linear(in_features=128, out_features=1, bias=True)
  (cls): Linear(in_features=128, out_features=1, bias=True)
)

AttentionMIL(
  (phi): Sequential(
    (0): Linear(in_features=562, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (attn): Linear(in_features=128, out_features=1, bias=True)
  (cls): Linear(in_features=128, out_features=1, bias=True)
)

...

AttentionMIL(
  (phi): Sequential(
    (0): Linear(in_features=562, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (attn): Linear(in_features=128, out_features=1, bias=True)
  (cls): Linear(in_features=128, out_features=1, bias=True)
)

[FA_PT_k=3] auc=0.850 balacc=0.798 acc=0.796 n_test=49


AttentionMIL(
  (phi): Sequential(
    (0): Linear(in_features=562, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (attn): Linear(in_features=128, out_features=1, bias=True)
  (cls): Linear(in_features=128, out_features=1, bias=True)
)

AttentionMIL(
  (phi): Sequential(
    (0): Linear(in_features=562, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (attn): Linear(in_features=128, out_features=1, bias=True)
  (cls): Linear(in_features=128, out_features=1, bias=True)
)

[FA_PT_k=4] auc=0.725 balacc=0.573 acc=0.569 n_test=51

=== CONCAT MIL (deep||concept) 5-fold summary (mean ± std) ===
           auc    balacc       acc
mean  0.810770  0.722772  0.718612
std   0.050175  0.090131  0.090603


gated mil

In [121]:
# ==========================================
# GATED MIL (Deep -> classifier, Concepts -> attention help)
# - loads patch-level concept CSV (with x,y, concept_*)
# - loads deep .pt per slide (features, coords)
# - aligns by coords
# - optional cap + topk selection (by deep-norm OR concept score)
# - 5-fold CV using your existing split folders
# ==========================================

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, accuracy_score
import h5py

# =========================
# CONFIG
# =========================
PATCH_CONCEPT_CSV = r"C:\Users\Vivian\Documents\CONCH\test_text_encoder\slide_concept_scores\noCAP_patch_ptfile_notopk_conf_PATCH.csv"
PT_FEAT_DIR       = r"C:\Users\Vivian\Documents\CONCH\conch_img_feats\10x_feats\40x\feats_pt"

FEAT_DIR = r"C:\Users\Vivian\Documents\CLAM\CLAM\FEATURES_DIR_5x\FEATURES_DIR_10x\uniextracted_mag10x_patch224_fp\feats_h5"
FEAT_EXT = ".h5"

CV_ROOT = r"C:/Users/Vivian/Documents/PANTHER/PANTHER/src/splits/cross-val"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 0

SLIDE_COL = "slide_id"
LABEL_COL = "true_label_str"
X_COL, Y_COL = "x", "y"
CONCEPT_PREFIX = "concept_"


# Bag controls
BAG_CAP = None            # None for no cap
TOPK = None               # None for no top-k
TOPK_MODE = "deep_norm"  # "deep_norm" or "concept_score"
CONCEPT_SCORE_MODE = "mean_all"  # "mean_all" or "mean_subset"
CONCEPT_SUBSET_IDXS = None       # e.g., [0,1,2] if you want subset scoring

# Train controls
EPOCHS = 8
LR = 1e-3
WEIGHT_DECAY = 1e-4
HID = 128
DROPOUT = 0.1

torch.manual_seed(SEED)
np.random.seed(SEED)

# =========================
# HELPERS
# =========================
def labels_to_int(arr):
    s = pd.Series(arr).astype(str).str.upper().str.strip()
    y = s.map({"FA": 0, "PT": 1})
    if y.isna().any():
        raise ValueError(f"Bad labels: {s[y.isna()].unique()}")
    return y.astype(int).to_numpy()

def read_slide_list(csv_path: str):
    d = pd.read_csv(csv_path)
    if "slide" in d.columns:
        slides = d["slide"].astype(str).tolist()
    elif "slide_id" in d.columns:
        slides = d["slide_id"].astype(str).tolist()
    else:
        slides = d.iloc[:, 0].astype(str).tolist()
    return [os.path.splitext(s.strip())[0] for s in slides]

def find_fold_dirs(cv_root: str):
    out = []
    for name in sorted(os.listdir(cv_root)):
        p = os.path.join(cv_root, name)
        if os.path.isdir(p) and all(os.path.isfile(os.path.join(p, f"{x}.csv")) for x in ["train","val","test"]):
            out.append(p)
    return out

def load_slide_pt(pt_path: str):
    obj = torch.load(pt_path, map_location="cpu", weights_only=True)
    feats = obj["features"].float()  # (N,D)
    coords = obj["coords"].int()     # (N,2)
    # feats = F.normalize(feats, dim=1)
    return feats.numpy(), coords.numpy()

def load_slide_h5(h5_path: str):
    with h5py.File(h5_path, "r") as f:
        # common keys
        if "features" in f:
            feats = f["features"][:]
        elif "feats" in f:
            feats = f["feats"][:]
        else:
            raise KeyError(f"No 'features' or 'feats' in {h5_path}. Keys={list(f.keys())}")

        if "coords" not in f:
            raise KeyError(f"No 'coords' in {h5_path}. Needed for gating/alignment. Keys={list(f.keys())}")
        coords = f["coords"][:]

    return feats.astype(np.float32), coords.astype(np.int32)

def eval_metrics(y_true, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(int)
    return {
        "auc": roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else np.nan,
        "balacc": balanced_accuracy_score(y_true, y_pred),
        "acc": accuracy_score(y_true, y_pred),
    }

# =========================
# DATASET
# =========================
class GatedBagDataset(Dataset):
    """
    Returns one slide (bag):
      Xd: (N, Ddeep)
      Xc: (N, Kconcept)
      y : scalar
    Alignment via (x,y) coords.
    """
    def __init__(
        self,
        df_patch_concept: pd.DataFrame,
        slide_ids,
        concept_cols,
        pt_dir,
        bag_cap=None,
        topk=None,
        topk_mode="deep_norm",
        concept_score_mode="mean_all",
        concept_subset_idxs=None,
        seed=0,
    ):
        self.df = df_patch_concept[df_patch_concept[SLIDE_COL].isin(slide_ids)].copy()
        self.slide_ids = [s for s in slide_ids if s in set(self.df[SLIDE_COL].unique())]
        self.concept_cols = concept_cols
        self.pt_dir = pt_dir

        self.bag_cap = bag_cap
        self.topk = topk
        self.topk_mode = topk_mode
        self.concept_score_mode = concept_score_mode
        self.concept_subset_idxs = concept_subset_idxs

        self.rng = np.random.default_rng(seed)

        # labels per slide
        lab = self.df.groupby(SLIDE_COL)[LABEL_COL].first()
        self.y_map = {sid: int(labels_to_int([lab.loc[sid]])[0]) for sid in self.slide_ids}

        # group for fast per-slide access
        self.by_slide = {sid: d for sid, d in self.df.groupby(SLIDE_COL)}

    def __len__(self):
        return len(self.slide_ids)

    def __getitem__(self, idx):
        sid = self.slide_ids[idx]

        # conch feats ----------
        pt_path = os.path.join(self.pt_dir, f"{sid}.pt")
        if not os.path.isfile(pt_path):
            raise FileNotFoundError(pt_path)

        E, coords = load_slide_pt(pt_path)  # E:(N,D), coords:(N,2)
        # -----------------------

        # uni feats h5 -----------
        # feat_path = os.path.join(self.pt_dir, f"{sid}{FEAT_EXT}")
        # if not os.path.isfile(feat_path):
        #     raise FileNotFoundError(feat_path)

        # E, coords = load_slide_h5(feat_path)
        # -----------------------

        d = self.by_slide[sid]

        # build concept map (x,y) -> concept vector
        keys = list(zip(d[X_COL].astype(int).to_numpy(), d[Y_COL].astype(int).to_numpy()))
        C = d[self.concept_cols].to_numpy(dtype=np.float32)
        concept_map = {k: C[i] for i, k in enumerate(keys)}

        # align concepts to deep coords
        Xc = np.zeros((coords.shape[0], len(self.concept_cols)), dtype=np.float32)
        for i, (x, y) in enumerate(coords.astype(int)):
            v = concept_map.get((x, y), None)
            if v is not None:
                Xc[i] = v

        Xd = E.astype(np.float32)

        # ---- selection indices (apply consistently to Xd/Xc) ----
        N = Xd.shape[0]
        sel = np.arange(N)

        # random cap first (optional)
        if self.bag_cap is not None and N > self.bag_cap:
            sel = self.rng.choice(N, size=self.bag_cap, replace=False)

        Xd = Xd[sel]
        Xc = Xc[sel]

        # attention mil - top-k by deep norm (optional, no conf needed)------
        # score = np.linalg.norm(Xd, axis=1)
        # keep = np.argsort(-score)[:200]
        # Xd = Xd[keep]
        # Xc = Xc[keep]
        #----------------------


        # top-k next (optional)
        if self.topk is not None and Xd.shape[0] > self.topk:
            if self.topk_mode == "deep_norm":
                score = np.linalg.norm(Xd, axis=1)  # (N,)
            elif self.topk_mode == "concept_score":
                if self.concept_score_mode == "mean_all":
                    score = Xc.mean(axis=1)
                elif self.concept_score_mode == "mean_subset":
                    if not self.concept_subset_idxs:
                        raise ValueError("concept_subset_idxs is required for mean_subset")
                    score = Xc[:, self.concept_subset_idxs].mean(axis=1)
                else:
                    raise ValueError(f"Bad concept_score_mode: {self.concept_score_mode}")
            else:
                raise ValueError(f"Bad topk_mode: {self.topk_mode}")

            keep = np.argsort(-score)[: self.topk]
            Xd = Xd[keep]
            Xc = Xc[keep]

        y = self.y_map[sid]
        return torch.from_numpy(Xd), torch.from_numpy(Xc), torch.tensor(y, dtype=torch.long), sid

def collate_bag(batch):
    return batch  # list of length B, each is one slide

# =========================
# MODEL
# =========================
class GatedAttentionMIL(nn.Module):
    """
    Attention logits a_i = a_deep(Hd_i) + a_concept(Hc_i)
    Pooled representation uses Hd (deep stream), then classify.
    """
    def __init__(self, deep_dim, concept_dim, hid=128, drop=0.1):
        super().__init__()
        self.phi_d = nn.Sequential(
            nn.Linear(deep_dim, hid),
            nn.ReLU(),
            nn.Dropout(drop),
        )
        self.phi_c = nn.Sequential(
            nn.Linear(concept_dim, hid),
            nn.ReLU(),
            nn.Dropout(drop),
        )
        self.attn_d = nn.Linear(hid, 1)
        self.attn_c = nn.Linear(hid, 1)
        self.cls = nn.Linear(hid, 1)

    def forward(self, Xd, Xc):
        Hd = self.phi_d(Xd)                 # (N,hid)
        Hc = self.phi_c(Xc)                 # (N,hid)
        a = self.attn_d(Hd).squeeze(1) + self.attn_c(Hc).squeeze(1)  # (N,)
        w = torch.softmax(a, dim=0)         # (N,)
        z = (w.unsqueeze(1) * Hd).sum(0)    # (hid,)
        logit = self.cls(z).squeeze(0)      # ()
        return logit, w

# class AttentionMIL(nn.Module):
#     def __init__(self, in_dim, hid=128, drop=0.1):
#         super().__init__()
#         self.phi = nn.Sequential(
#             nn.Linear(in_dim, hid),
#             nn.ReLU(),
#             nn.Dropout(drop),
#         )
#         self.attn = nn.Linear(hid, 1)
#         self.cls = nn.Linear(hid, 1)

#     def forward(self, X):  # X:(N,D)
#         H = self.phi(X)                 # (N,hid)
#         a = self.attn(H).squeeze(1)     # (N,)
#         w = torch.softmax(a, dim=0)     # (N,)
#         z = (w.unsqueeze(1) * H).sum(0) # (hid,)
#         logit = self.cls(z).squeeze(0)
#         return logit, w

# =========================
# LOAD CSV
# =========================
df_patch = pd.read_csv(PATCH_CONCEPT_CSV)
df_patch.columns = [c.strip() for c in df_patch.columns]

for col in [SLIDE_COL, LABEL_COL, X_COL, Y_COL]:
    if col not in df_patch.columns:
        raise ValueError(f"Missing required column '{col}' in PATCH_CONCEPT_CSV")

concept_cols = [c for c in df_patch.columns if c.startswith(CONCEPT_PREFIX)]
if not concept_cols:
    raise ValueError("No concept_* columns found in PATCH_CONCEPT_CSV")

# numeric + impute
df_patch[concept_cols] = df_patch[concept_cols].apply(pd.to_numeric, errors="coerce")
df_patch[concept_cols] = df_patch[concept_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)

fold_dirs = find_fold_dirs(CV_ROOT)

# =========================
# 5-FOLD CV
# =========================
all_fold = []

for fold_dir in fold_dirs:
    fold_name = os.path.basename(fold_dir)

    train_slides = read_slide_list(os.path.join(fold_dir, "train.csv"))
    val_slides   = read_slide_list(os.path.join(fold_dir, "val.csv"))
    test_slides  = read_slide_list(os.path.join(fold_dir, "test.csv"))
    fit_slides = train_slides + val_slides  # simple

    # replace PT_FEAT_DIR with FEAT_DIR and load from .h5 instead of .pt
    ds_tr = GatedBagDataset(
        df_patch, fit_slides, concept_cols, PT_FEAT_DIR,
        bag_cap=BAG_CAP, topk=TOPK, topk_mode=TOPK_MODE,
        concept_score_mode=CONCEPT_SCORE_MODE, concept_subset_idxs=CONCEPT_SUBSET_IDXS,
        seed=SEED
    )
    ds_te = GatedBagDataset(
        df_patch, test_slides, concept_cols, PT_FEAT_DIR,
        bag_cap=BAG_CAP, topk=TOPK, topk_mode=TOPK_MODE,
        concept_score_mode=CONCEPT_SCORE_MODE, concept_subset_idxs=CONCEPT_SUBSET_IDXS,
        seed=SEED
    )
    

    dl_tr = DataLoader(ds_tr, batch_size=1, shuffle=True, collate_fn=collate_bag)
    dl_te = DataLoader(ds_te, batch_size=1, shuffle=False, collate_fn=collate_bag)

    # infer dims
    Xd0, Xc0, _, _ = ds_tr[0]
    model = GatedAttentionMIL(deep_dim=Xd0.shape[1], concept_dim=Xc0.shape[1], hid=HID, drop=DROPOUT).to(DEVICE)
    
    # deep feat only (ablation)
    # Xd0, _, _, _ = ds_tr[0]   # ignore concept
    # model = AttentionMIL(in_dim=Xd0.shape[1], hid=HID, drop=DROPOUT).to(DEVICE)


    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.BCEWithLogitsLoss()

    # train
    model.train()
    for ep in range(EPOCHS):
        for batch in dl_tr:
            (Xd, Xc, y, sid) = batch[0]
            Xd = Xd.to(DEVICE)
            Xc = Xc.to(DEVICE)
            y  = y.float().to(DEVICE)

            logit, _ = model(Xd, Xc)
            # logit, w = model(Xd) # deep feat only

            loss = loss_fn(logit.view(1), y.view(1))

            opt.zero_grad()
            loss.backward()
            opt.step()

    # eval
    model.eval()
    y_true, y_prob = [], []
    with torch.no_grad():
        for batch in dl_te:
            (Xd, Xc, y, sid) = batch[0]
            Xd = Xd.to(DEVICE)
            Xc = Xc.to(DEVICE)
            logit, w = model(Xd, Xc)
            # logit, _ = model(Xd) # deep feat only

            prob = torch.sigmoid(logit).item()
            y_true.append(int(y.item()))
            y_prob.append(prob)

    y_true = np.array(y_true)
    y_prob = np.array(y_prob)
    m = eval_metrics(y_true, y_prob, thr=0.5)

    all_fold.append({"fold": fold_name, **m, "n_test": len(y_true)})
    print(f"[{fold_name}] auc={m['auc']:.3f} balacc={m['balacc']:.3f} acc={m['acc']:.3f} (n_test={len(y_true)})")

res = pd.DataFrame(all_fold)
print("\n=== GATED MIL (deep clf, concept-guided attention) 5-fold summary (mean ± std) ===")
print(res[["auc","balacc","acc"]].agg(["mean","std"]))


GatedAttentionMIL(
  (phi_d): Sequential(
    (0): Linear(in_features=512, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (phi_c): Sequential(
    (0): Linear(in_features=50, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (attn_d): Linear(in_features=128, out_features=1, bias=True)
  (attn_c): Linear(in_features=128, out_features=1, bias=True)
  (cls): Linear(in_features=128, out_features=1, bias=True)
)

GatedAttentionMIL(
  (phi_d): Sequential(
    (0): Linear(in_features=512, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (phi_c): Sequential(
    (0): Linear(in_features=50, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (attn_d): Linear(in_features=128, out_features=1, bias=True)
  (attn_c): Linear(in_features=128, out_features=1, bias=True)
  (cls): Linear(in_features=128, out_features=1, bias=True)
)

[FA_PT_k=0] auc=0.813 balacc=0.752 acc=0.745 (n_test=47)


GatedAttentionMIL(
  (phi_d): Sequential(
    (0): Linear(in_features=512, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (phi_c): Sequential(
    (0): Linear(in_features=50, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (attn_d): Linear(in_features=128, out_features=1, bias=True)
  (attn_c): Linear(in_features=128, out_features=1, bias=True)
  (cls): Linear(in_features=128, out_features=1, bias=True)
)

GatedAttentionMIL(
  (phi_d): Sequential(
    (0): Linear(in_features=512, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (phi_c): Sequential(
    (0): Linear(in_features=50, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (attn_d): Linear(in_features=128, out_features=1, bias=True)
  (attn_c): Linear(in_features=128, out_features=1, bias=True)
  (cls): Linear(in_features=128, out_features=1, bias=True)
)

...

GatedAttentionMIL(
  (phi_d): Sequential(
    (0): Linear(in_features=512, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (phi_c): Sequential(
    (0): Linear(in_features=50, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (attn_d): Linear(in_features=128, out_features=1, bias=True)
  (attn_c): Linear(in_features=128, out_features=1, bias=True)
  (cls): Linear(in_features=128, out_features=1, bias=True)
)

[FA_PT_k=3] auc=0.867 balacc=0.732 acc=0.735 (n_test=49)


GatedAttentionMIL(
  (phi_d): Sequential(
    (0): Linear(in_features=512, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (phi_c): Sequential(
    (0): Linear(in_features=50, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (attn_d): Linear(in_features=128, out_features=1, bias=True)
  (attn_c): Linear(in_features=128, out_features=1, bias=True)
  (cls): Linear(in_features=128, out_features=1, bias=True)
)

GatedAttentionMIL(
  (phi_d): Sequential(
    (0): Linear(in_features=512, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (phi_c): Sequential(
    (0): Linear(in_features=50, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (attn_d): Linear(in_features=128, out_features=1, bias=True)
  (attn_c): Linear(in_features=128, out_features=1, bias=True)
  (cls): Linear(in_features=128, out_features=1, bias=True)
)

[FA_PT_k=4] auc=0.738 balacc=0.687 acc=0.686 (n_test=51)

=== GATED MIL (deep clf, concept-guided attention) 5-fold summary (mean ± std) ===
           auc    balacc       acc
mean  0.819571  0.727951  0.725630
std   0.049189  0.025350  0.022725


5x + 10x concepts

In [113]:
# ==========================================
# GATED MIL (Deep -> classifier, Concepts -> attention help)
# - loads patch-level concept CSV (with x,y, concept_*)
# - loads deep .pt per slide (features, coords)
# - aligns by coords
# - optional cap + topk selection (by deep-norm OR concept score)
# - 5-fold CV using your existing split folders
# ==========================================

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, accuracy_score

# =========================
# CONFIG
# =========================
PATCH_CONCEPT_CSV = r"C:\Users\Vivian\Documents\CONCH\test_text_encoder\slide_concept_scores\noCAP_patch_ptfile_notopk_conf_PATCH.csv"
PT_FEAT_DIR       = r"C:\Users\Vivian\Documents\CONCH\conch_img_feats\10x_feats\40x\pt"
CV_ROOT           = r"C:/Users/Vivian/Documents/PANTHER/PANTHER/src/splits/cross-val"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 0

SLIDE_COL = "slide_id"
LABEL_COL = "true_label_str"
X_COL, Y_COL = "x", "y"
CONCEPT_PREFIX = "concept_"
# coords mapping: deep grid -> concept grid
COORD_SCALE = 1   # 5x -> 10x


# Bag controls
BAG_CAP = None            # None for no cap
TOPK = None               # None for no top-k
TOPK_MODE = "deep_norm"  # "deep_norm" or "concept_score"
CONCEPT_SCORE_MODE = "mean_all"  # "mean_all" or "mean_subset"
CONCEPT_SUBSET_IDXS = None       # e.g., [0,1,2] if you want subset scoring

# Train controls
EPOCHS = 8
LR = 1e-3
WEIGHT_DECAY = 1e-4
HID = 128
DROPOUT = 0.1

torch.manual_seed(SEED)
np.random.seed(SEED)

# =========================
# HELPERS
# =========================
def labels_to_int(arr):
    s = pd.Series(arr).astype(str).str.upper().str.strip()
    y = s.map({"FA": 0, "PT": 1})
    if y.isna().any():
        raise ValueError(f"Bad labels: {s[y.isna()].unique()}")
    return y.astype(int).to_numpy()

def read_slide_list(csv_path: str):
    d = pd.read_csv(csv_path)
    if "slide" in d.columns:
        slides = d["slide"].astype(str).tolist()
    elif "slide_id" in d.columns:
        slides = d["slide_id"].astype(str).tolist()
    else:
        slides = d.iloc[:, 0].astype(str).tolist()
    return [os.path.splitext(s.strip())[0] for s in slides]

def find_fold_dirs(cv_root: str):
    out = []
    for name in sorted(os.listdir(cv_root)):
        p = os.path.join(cv_root, name)
        if os.path.isdir(p) and all(os.path.isfile(os.path.join(p, f"{x}.csv")) for x in ["train","val","test"]):
            out.append(p)
    return out

def load_slide_pt(pt_path: str):
    obj = torch.load(pt_path, map_location="cpu", weights_only=True)
    feats = obj["features"].float()  # (N,D)
    coords = obj["coords"].int()     # (N,2)
    # feats = F.normalize(feats, dim=1)
    return feats.numpy(), coords.numpy()

def eval_metrics(y_true, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(int)
    return {
        "auc": roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else np.nan,
        "balacc": balanced_accuracy_score(y_true, y_pred),
        "acc": accuracy_score(y_true, y_pred),
    }

# =========================
# DATASET
# =========================
class GatedBagDataset(Dataset):
    """
    Returns one slide (bag):
      Xd: (N, Ddeep)
      Xc: (N, Kconcept)
      y : scalar
    Alignment via (x,y) coords.
    """
    def __init__(
        self,
        df_patch_concept: pd.DataFrame,
        slide_ids,
        concept_cols,
        pt_dir,
        bag_cap=None,
        topk=None,
        topk_mode="deep_norm",
        concept_score_mode="mean_all",
        concept_subset_idxs=None,
        seed=0,
    ):
        self.df = df_patch_concept[df_patch_concept[SLIDE_COL].isin(slide_ids)].copy()
        self.slide_ids = [s for s in slide_ids if s in set(self.df[SLIDE_COL].unique())]
        self.concept_cols = concept_cols
        self.pt_dir = pt_dir

        self.bag_cap = bag_cap
        self.topk = topk
        self.topk_mode = topk_mode
        self.concept_score_mode = concept_score_mode
        self.concept_subset_idxs = concept_subset_idxs

        self.rng = np.random.default_rng(seed)

        # labels per slide
        lab = self.df.groupby(SLIDE_COL)[LABEL_COL].first()
        self.y_map = {sid: int(labels_to_int([lab.loc[sid]])[0]) for sid in self.slide_ids}

        # group for fast per-slide access
        self.by_slide = {sid: d for sid, d in self.df.groupby(SLIDE_COL)}

    def __len__(self):
        return len(self.slide_ids)

    def __getitem__(self, idx):
        sid = self.slide_ids[idx]
        pt_path = os.path.join(self.pt_dir, f"{sid}.pt")
        if not os.path.isfile(pt_path):
            raise FileNotFoundError(pt_path)

        E, coords = load_slide_pt(pt_path)  # E:(N,D), coords:(N,2)
        d = self.by_slide[sid]

        # build concept map (x,y) -> concept vector
        keys = list(zip(d[X_COL].astype(int).to_numpy(), d[Y_COL].astype(int).to_numpy()))
        C = d[self.concept_cols].to_numpy(dtype=np.float32)
        concept_map = {k: C[i] for i, k in enumerate(keys)}

        # align concepts to deep coords
        # build concept map (x,y) -> concept vector
        d = self.by_slide[sid]
        keys = list(zip(d[X_COL].astype(int).to_numpy(), d[Y_COL].astype(int).to_numpy()))
        C = d[self.concept_cols].to_numpy(dtype=np.float32)
        concept_map = {k: C[i] for i, k in enumerate(keys)}

        Xd = E.astype(np.float32)
        coords5 = coords.astype(int)

        # map 5x coords into 10x coord system for lookup
        coords10 = coords5 * COORD_SCALE

        # keep only deep patches that have a concept match
        matched_idx = []
        Xc_list = []

        for i, (x, y) in enumerate(coords10):
            v = concept_map.get((int(x), int(y)), None)
            if v is not None:
                matched_idx.append(i)
                Xc_list.append(v)

        if len(matched_idx) == 0:
            # fallback: keep a tiny bag with zeros so training doesn't crash
            # (but ideally you log this and fix coordinate assumptions)
            Xd = Xd[:1]
            Xc = np.zeros((1, len(self.concept_cols)), dtype=np.float32)
        else:
            Xd = Xd[matched_idx]
            Xc = np.stack(Xc_list, axis=0).astype(np.float32)


        # ---- selection indices (apply consistently to Xd/Xc) ----
        N = Xd.shape[0]
        sel = np.arange(N)

        # random cap first (optional)
        if self.bag_cap is not None and N > self.bag_cap:
            sel = self.rng.choice(N, size=self.bag_cap, replace=False)

        Xd = Xd[sel]
        Xc = Xc[sel]

        # attention mil - top-k by deep norm (optional, no conf needed)------
        # score = np.linalg.norm(Xd, axis=1)
        # keep = np.argsort(-score)[:200]
        # Xd = Xd[keep]
        # Xc = Xc[keep]
        #----------------------


        # top-k next (optional)
        if self.topk is not None and Xd.shape[0] > self.topk:
            if self.topk_mode == "deep_norm":
                score = np.linalg.norm(Xd, axis=1)  # (N,)
            elif self.topk_mode == "concept_score":
                if self.concept_score_mode == "mean_all":
                    score = Xc.mean(axis=1)
                elif self.concept_score_mode == "mean_subset":
                    if not self.concept_subset_idxs:
                        raise ValueError("concept_subset_idxs is required for mean_subset")
                    score = Xc[:, self.concept_subset_idxs].mean(axis=1)
                else:
                    raise ValueError(f"Bad concept_score_mode: {self.concept_score_mode}")
            else:
                raise ValueError(f"Bad topk_mode: {self.topk_mode}")

            keep = np.argsort(-score)[: self.topk]
            Xd = Xd[keep]
            Xc = Xc[keep]

        y = self.y_map[sid]
        return torch.from_numpy(Xd), torch.from_numpy(Xc), torch.tensor(y, dtype=torch.long), sid

def collate_bag(batch):
    return batch  # list of length B, each is one slide

# =========================
# MODEL
# =========================
class GatedAttentionMIL(nn.Module):
    """
    Attention logits a_i = a_deep(Hd_i) + a_concept(Hc_i)
    Pooled representation uses Hd (deep stream), then classify.
    """
    def __init__(self, deep_dim, concept_dim, hid=128, drop=0.1):
        super().__init__()
        self.phi_d = nn.Sequential(
            nn.Linear(deep_dim, hid),
            nn.ReLU(),
            nn.Dropout(drop),
        )
        self.phi_c = nn.Sequential(
            nn.Linear(concept_dim, hid),
            nn.ReLU(),
            nn.Dropout(drop),
        )
        self.attn_d = nn.Linear(hid, 1)
        self.attn_c = nn.Linear(hid, 1)
        self.cls = nn.Linear(hid, 1)

    def forward(self, Xd, Xc):
        Hd = self.phi_d(Xd)                 # (N,hid)
        Hc = self.phi_c(Xc)                 # (N,hid)
        a = self.attn_d(Hd).squeeze(1) + self.attn_c(Hc).squeeze(1)  # (N,)
        w = torch.softmax(a, dim=0)         # (N,)
        z = (w.unsqueeze(1) * Hd).sum(0)    # (hid,)
        logit = self.cls(z).squeeze(0)      # ()
        return logit, w

# class AttentionMIL(nn.Module):
#     def __init__(self, in_dim, hid=128, drop=0.1):
#         super().__init__()
#         self.phi = nn.Sequential(
#             nn.Linear(in_dim, hid),
#             nn.ReLU(),
#             nn.Dropout(drop),
#         )
#         self.attn = nn.Linear(hid, 1)
#         self.cls = nn.Linear(hid, 1)

#     def forward(self, X):  # X:(N,D)
#         H = self.phi(X)                 # (N,hid)
#         a = self.attn(H).squeeze(1)     # (N,)
#         w = torch.softmax(a, dim=0)     # (N,)
#         z = (w.unsqueeze(1) * H).sum(0) # (hid,)
#         logit = self.cls(z).squeeze(0)
#         return logit, w

# =========================
# LOAD CSV
# =========================
df_patch = pd.read_csv(PATCH_CONCEPT_CSV)
df_patch.columns = [c.strip() for c in df_patch.columns]

for col in [SLIDE_COL, LABEL_COL, X_COL, Y_COL]:
    if col not in df_patch.columns:
        raise ValueError(f"Missing required column '{col}' in PATCH_CONCEPT_CSV")

concept_cols = [c for c in df_patch.columns if c.startswith(CONCEPT_PREFIX)]
if not concept_cols:
    raise ValueError("No concept_* columns found in PATCH_CONCEPT_CSV")

# numeric + impute
df_patch[concept_cols] = df_patch[concept_cols].apply(pd.to_numeric, errors="coerce")
df_patch[concept_cols] = df_patch[concept_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)

fold_dirs = find_fold_dirs(CV_ROOT)

# =========================
# 5-FOLD CV
# =========================
all_fold = []

for fold_dir in fold_dirs:
    fold_name = os.path.basename(fold_dir)

    train_slides = read_slide_list(os.path.join(fold_dir, "train.csv"))
    val_slides   = read_slide_list(os.path.join(fold_dir, "val.csv"))
    test_slides  = read_slide_list(os.path.join(fold_dir, "test.csv"))
    fit_slides = train_slides + val_slides  # simple

    ds_tr = GatedBagDataset(
        df_patch, fit_slides, concept_cols, PT_FEAT_DIR,
        bag_cap=BAG_CAP, topk=TOPK, topk_mode=TOPK_MODE,
        concept_score_mode=CONCEPT_SCORE_MODE, concept_subset_idxs=CONCEPT_SUBSET_IDXS,
        seed=SEED
    )
    ds_te = GatedBagDataset(
        df_patch, test_slides, concept_cols, PT_FEAT_DIR,
        bag_cap=BAG_CAP, topk=TOPK, topk_mode=TOPK_MODE,
        concept_score_mode=CONCEPT_SCORE_MODE, concept_subset_idxs=CONCEPT_SUBSET_IDXS,
        seed=SEED
    )

    dl_tr = DataLoader(ds_tr, batch_size=1, shuffle=True, collate_fn=collate_bag)
    dl_te = DataLoader(ds_te, batch_size=1, shuffle=False, collate_fn=collate_bag)

    # infer dims
    Xd0, Xc0, _, _ = ds_tr[0]
    model = GatedAttentionMIL(deep_dim=Xd0.shape[1], concept_dim=Xc0.shape[1], hid=HID, drop=DROPOUT).to(DEVICE)
    
    # deep feat only (ablation)
    # Xd0, _, _, _ = ds_tr[0]   # ignore concept
    # model = AttentionMIL(in_dim=Xd0.shape[1], hid=HID, drop=DROPOUT).to(DEVICE)


    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.BCEWithLogitsLoss()

    # train
    model.train()
    for ep in range(EPOCHS):
        for batch in dl_tr:
            (Xd, Xc, y, sid) = batch[0]
            Xd = Xd.to(DEVICE)
            Xc = Xc.to(DEVICE)
            y  = y.float().to(DEVICE)

            logit, _ = model(Xd, Xc)
            # logit, w = model(Xd) # deep feat only

            loss = loss_fn(logit.view(1), y.view(1))

            opt.zero_grad()
            loss.backward()
            opt.step()

    # eval
    model.eval()
    y_true, y_prob = [], []
    with torch.no_grad():
        for batch in dl_te:
            (Xd, Xc, y, sid) = batch[0]
            Xd = Xd.to(DEVICE)
            Xc = Xc.to(DEVICE)
            logit, w = model(Xd, Xc)
            # logit, _ = model(Xd) # deep feat only

            prob = torch.sigmoid(logit).item()
            y_true.append(int(y.item()))
            y_prob.append(prob)

    y_true = np.array(y_true)
    y_prob = np.array(y_prob)
    m = eval_metrics(y_true, y_prob, thr=0.5)

    all_fold.append({"fold": fold_name, **m, "n_test": len(y_true)})
    print(f"[{fold_name}] auc={m['auc']:.3f} balacc={m['balacc']:.3f} acc={m['acc']:.3f} (n_test={len(y_true)})")

res = pd.DataFrame(all_fold)
print("\n=== GATED MIL (deep clf, concept-guided attention) 5-fold summary (mean ± std) ===")
print(res[["auc","balacc","acc"]].agg(["mean","std"]))


GatedAttentionMIL(
  (phi_d): Sequential(
    (0): Linear(in_features=512, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (phi_c): Sequential(
    (0): Linear(in_features=50, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (attn_d): Linear(in_features=128, out_features=1, bias=True)
  (attn_c): Linear(in_features=128, out_features=1, bias=True)
  (cls): Linear(in_features=128, out_features=1, bias=True)
)

GatedAttentionMIL(
  (phi_d): Sequential(
    (0): Linear(in_features=512, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (phi_c): Sequential(
    (0): Linear(in_features=50, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (attn_d): Linear(in_features=128, out_features=1, bias=True)
  (attn_c): Linear(in_features=128, out_features=1, bias=True)
  (cls): Linear(in_features=128, out_features=1, bias=True)
)

[FA_PT_k=0] auc=0.813 balacc=0.752 acc=0.745 (n_test=47)


GatedAttentionMIL(
  (phi_d): Sequential(
    (0): Linear(in_features=512, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (phi_c): Sequential(
    (0): Linear(in_features=50, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (attn_d): Linear(in_features=128, out_features=1, bias=True)
  (attn_c): Linear(in_features=128, out_features=1, bias=True)
  (cls): Linear(in_features=128, out_features=1, bias=True)
)

GatedAttentionMIL(
  (phi_d): Sequential(
    (0): Linear(in_features=512, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (phi_c): Sequential(
    (0): Linear(in_features=50, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (attn_d): Linear(in_features=128, out_features=1, bias=True)
  (attn_c): Linear(in_features=128, out_features=1, bias=True)
  (cls): Linear(in_features=128, out_features=1, bias=True)
)

...

GatedAttentionMIL(
  (phi_d): Sequential(
    (0): Linear(in_features=512, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (phi_c): Sequential(
    (0): Linear(in_features=50, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (attn_d): Linear(in_features=128, out_features=1, bias=True)
  (attn_c): Linear(in_features=128, out_features=1, bias=True)
  (cls): Linear(in_features=128, out_features=1, bias=True)
)

[FA_PT_k=3] auc=0.867 balacc=0.732 acc=0.735 (n_test=49)


GatedAttentionMIL(
  (phi_d): Sequential(
    (0): Linear(in_features=512, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (phi_c): Sequential(
    (0): Linear(in_features=50, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (attn_d): Linear(in_features=128, out_features=1, bias=True)
  (attn_c): Linear(in_features=128, out_features=1, bias=True)
  (cls): Linear(in_features=128, out_features=1, bias=True)
)

GatedAttentionMIL(
  (phi_d): Sequential(
    (0): Linear(in_features=512, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (phi_c): Sequential(
    (0): Linear(in_features=50, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (attn_d): Linear(in_features=128, out_features=1, bias=True)
  (attn_c): Linear(in_features=128, out_features=1, bias=True)
  (cls): Linear(in_features=128, out_features=1, bias=True)
)

[FA_PT_k=4] auc=0.738 balacc=0.687 acc=0.686 (n_test=51)

=== GATED MIL (deep clf, concept-guided attention) 5-fold summary (mean ± std) ===
           auc    balacc       acc
mean  0.819571  0.727951  0.725630
std   0.049189  0.025350  0.022725


deep feats only mil

In [118]:
# DEEP-ONLY MIL BASELINE (Attention MIL)
# - loads deep .pt per slide (features, coords)
# - (no concepts, no gating)
# - optional cap + top-k selection (by deep-norm)
# - 5-fold CV using your existing split folders
# ==========================================

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, accuracy_score

# =========================
# CONFIG
# =========================
# PT_FEAT_DIR = r"C:\Users\Vivian\Documents\CONCH\conch_img_feats\10x_feats\40x\pt"  # <-- set this
PT_FEAT_DIR = r'C:\Users\Vivian\Documents\CLAM\CLAM\FEATURES_DIR_5x\FEATURES_DIR_10x\uniextracted_mag10x_patch224_fp\feats_pt' # uni feats
CV_ROOT     = r"C:/Users/Vivian/Documents/PANTHER/PANTHER/src/splits/cross-val"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 0

# Bag controls
BAG_CAP   = None     # None for no random cap
TOPK      = None     # None for no top-k
TOPK_MODE = "deep_norm"  # only deep_norm here

# Train controls
EPOCHS = 8
LR = 1e-3
WEIGHT_DECAY = 1e-4
HID = 128
DROPOUT = 0.1

torch.manual_seed(SEED)
np.random.seed(SEED)

# =========================
# HELPERS
# =========================
def labels_to_int(arr):
    s = pd.Series(arr).astype(str).str.upper().str.strip()
    y = s.map({"FA": 0, "PT": 1})
    if y.isna().any():
        raise ValueError(f"Bad labels: {s[y.isna()].unique()}")
    return y.astype(int).to_numpy()

def read_slide_list(csv_path: str):
    d = pd.read_csv(csv_path)
    if "slide" in d.columns:
        slides = d["slide"].astype(str).tolist()
    elif "slide_id" in d.columns:
        slides = d["slide_id"].astype(str).tolist()
    else:
        slides = d.iloc[:, 0].astype(str).tolist()
    return [os.path.splitext(s.strip())[0] for s in slides]

def find_fold_dirs(cv_root: str):
    out = []
    for name in sorted(os.listdir(cv_root)):
        p = os.path.join(cv_root, name)
        if os.path.isdir(p) and all(os.path.isfile(os.path.join(p, f"{x}.csv")) for x in ["train", "val", "test"]):
            out.append(p)
    return out

# def load_slide_pt(pt_path: str):
#     obj = torch.load(pt_path, map_location="cpu", weights_only=True)
#     feats = obj["features"].float()  # (N,D)
#     coords = obj.get("coords", None)
#     if coords is None:
#         coords = torch.zeros((feats.shape[0], 2), dtype=torch.int32)
#     return feats.numpy(), coords.numpy()

def load_slide_pt(pt_path: str):
    obj = torch.load(pt_path, map_location="cpu")  # don't use weights_only here

    # Case A: saved as dict
    if isinstance(obj, dict):
        if "features" in obj:
            feats = obj["features"]
        elif "feats" in obj:
            feats = obj["feats"]
        else:
            # fallback: take first tensor-like value
            tensor_vals = [v for v in obj.values() if torch.is_tensor(v)]
            if not tensor_vals:
                raise ValueError(f"No tensor found in dict .pt: {pt_path}, keys={list(obj.keys())}")
            feats = tensor_vals[0]

    # Case B: saved as tensor directly
    elif torch.is_tensor(obj):
        feats = obj

    # Case C: saved as numpy
    elif isinstance(obj, np.ndarray):
        feats = torch.from_numpy(obj)

    else:
        raise ValueError(f"Unsupported .pt content type: {type(obj)} in {pt_path}")

    feats = feats.float()

    # return feats only; coords are unused in deep-only baseline
    return feats.numpy(), None


def eval_metrics(y_true, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(int)
    return {
        "auc": roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else np.nan,
        "balacc": balanced_accuracy_score(y_true, y_pred),
        "acc": accuracy_score(y_true, y_pred),
    }

def derive_label_from_slide_id(slide_id: str) -> int:
    s = str(slide_id).upper()
    if s.startswith("FA"):
        return 0
    if s.startswith("PT"):
        return 1
    raise ValueError(f"Can't derive label from slide_id='{slide_id}'. Add labels or adjust parsing.")

# =========================
# DATASET
# =========================
class DeepBagDataset(Dataset):
    """
    Returns one slide (bag):
      Xd: (N, Ddeep)
      y : scalar
    """
    def __init__(
        self,
        slide_ids,
        pt_dir,
        bag_cap=None,
        topk=None,
        seed=0,
    ):
        self.slide_ids = slide_ids
        self.pt_dir = pt_dir
        self.bag_cap = bag_cap
        self.topk = topk
        self.rng = np.random.default_rng(seed)

        # label per slide (from slide_id prefix FA/PT)
        self.y_map = {sid: derive_label_from_slide_id(sid) for sid in self.slide_ids}

        # filter slides that actually exist on disk
        keep = []
        for sid in self.slide_ids:
            pt_path = os.path.join(self.pt_dir, f"{sid}.pt")
            if os.path.isfile(pt_path):
                keep.append(sid)
        self.slide_ids = keep

    def __len__(self):
        return len(self.slide_ids)

    def __getitem__(self, idx):
        sid = self.slide_ids[idx]
        pt_path = os.path.join(self.pt_dir, f"{sid}.pt")
        if not os.path.isfile(pt_path):
            raise FileNotFoundError(pt_path)

        Xd, _coords = load_slide_pt(pt_path)  # Xd:(N,D)
        Xd = Xd.astype(np.float32)
        N = Xd.shape[0]

        sel = np.arange(N)

        # random cap (optional)
        if self.bag_cap is not None and N > self.bag_cap:
            sel = self.rng.choice(N, size=self.bag_cap, replace=False)

        Xd = Xd[sel]

        # top-k by deep norm (optional)
        if self.topk is not None and Xd.shape[0] > self.topk:
            score = np.linalg.norm(Xd, axis=1)
            keep = np.argsort(-score)[: self.topk]
            Xd = Xd[keep]

        y = self.y_map[sid]
        return torch.from_numpy(Xd), torch.tensor(y, dtype=torch.long), sid

def collate_bag(batch):
    return batch  # list of length B, each is one slide

# =========================
# MODEL
# =========================
class AttentionMIL(nn.Module):
    def __init__(self, in_dim, hid=128, drop=0.1):
        super().__init__()
        self.phi = nn.Sequential(
            nn.Linear(in_dim, hid),
            nn.ReLU(),
            nn.Dropout(drop),
        )
        self.attn = nn.Linear(hid, 1)
        self.cls = nn.Linear(hid, 1)

    def forward(self, X):  # X:(N,D)
        H = self.phi(X)                 # (N,hid)
        a = self.attn(H).squeeze(1)     # (N,)
        w = torch.softmax(a, dim=0)     # (N,)
        z = (w.unsqueeze(1) * H).sum(0) # (hid,)
        logit = self.cls(z).squeeze(0)
        return logit, w

# =========================
# 5-FOLD CV
# =========================
fold_dirs = find_fold_dirs(CV_ROOT)
all_fold = []

for fold_dir in fold_dirs:
    fold_name = os.path.basename(fold_dir)

    train_slides = read_slide_list(os.path.join(fold_dir, "train.csv"))
    val_slides   = read_slide_list(os.path.join(fold_dir, "val.csv"))
    test_slides  = read_slide_list(os.path.join(fold_dir, "test.csv"))

    fit_slides = train_slides + val_slides

    ds_tr = DeepBagDataset(
        slide_ids=fit_slides,
        pt_dir=PT_FEAT_DIR,
        bag_cap=BAG_CAP,
        topk=TOPK,
        seed=SEED
    )
    ds_te = DeepBagDataset(
        slide_ids=test_slides,
        pt_dir=PT_FEAT_DIR,
        bag_cap=BAG_CAP,
        topk=TOPK,
        seed=SEED
    )

    dl_tr = DataLoader(ds_tr, batch_size=1, shuffle=True, collate_fn=collate_bag)
    dl_te = DataLoader(ds_te, batch_size=1, shuffle=False, collate_fn=collate_bag)

    if len(ds_tr) == 0 or len(ds_te) == 0:
        print(f"[{fold_name}] skipped (empty train/test after filtering missing .pt files)")
        continue

    # infer dim
    Xd0, _, _ = ds_tr[0]
    model = AttentionMIL(in_dim=Xd0.shape[1], hid=HID, drop=DROPOUT).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.BCEWithLogitsLoss()

    # train
    model.train()
    for ep in range(EPOCHS):
        for batch in dl_tr:
            (Xd, y, sid) = batch[0]
            Xd = Xd.to(DEVICE)
            y  = y.float().to(DEVICE)

            logit, _w = model(Xd)
            loss = loss_fn(logit.view(1), y.view(1))

            opt.zero_grad()
            loss.backward()
            opt.step()

    # eval
    model.eval()
    y_true, y_prob = [], []
    with torch.no_grad():
        for batch in dl_te:
            (Xd, y, sid) = batch[0]
            Xd = Xd.to(DEVICE)

            logit, _w = model(Xd)
            prob = torch.sigmoid(logit).item()

            y_true.append(int(y.item()))
            y_prob.append(prob)

    y_true = np.array(y_true)
    y_prob = np.array(y_prob)
    m = eval_metrics(y_true, y_prob, thr=0.5)

    all_fold.append({"fold": fold_name, **m, "n_test": len(y_true)})
    print(f"[{fold_name}] auc={m['auc']:.3f} balacc={m['balacc']:.3f} acc={m['acc']:.3f} (n_test={len(y_true)})")

res = pd.DataFrame(all_fold)
print("\n=== DEEP-ONLY Attention MIL 5-fold summary (mean ± std) ===")
print(res[["auc", "balacc", "acc"]].agg(["mean", "std"]))


C:\Users\Vivian\AppData\Local\Temp\ipykernel_35164\1723634624.py:78: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  obj = torch.load(pt_path, map_location="cpu")  # don't use

AttentionMIL(
  (phi): Sequential(
    (0): Linear(in_features=1024, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (attn): Linear(in_features=128, out_features=1, bias=True)
  (cls): Linear(in_features=128, out_features=1, bias=True)
)

AttentionMIL(
  (phi): Sequential(
    (0): Linear(in_features=1024, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (attn): Linear(in_features=128, out_features=1, bias=True)
  (cls): Linear(in_features=128, out_features=1, bias=True)
)

[FA_PT_k=0] auc=0.513 balacc=0.515 acc=0.489 (n_test=47)


C:\Users\Vivian\AppData\Local\Temp\ipykernel_35164\1723634624.py:78: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  obj = torch.load(pt_path, map_location="cpu")  # don't use

...

[FA_PT_k=3] auc=0.828 balacc=0.646 acc=0.653 (n_test=49)


C:\Users\Vivian\AppData\Local\Temp\ipykernel_35164\1723634624.py:78: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  obj = torch.load(pt_path, map_location="cpu")  # don't use

AttentionMIL(
  (phi): Sequential(
    (0): Linear(in_features=1024, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (attn): Linear(in_features=128, out_features=1, bias=True)
  (cls): Linear(in_features=128, out_features=1, bias=True)
)

AttentionMIL(
  (phi): Sequential(
    (0): Linear(in_features=1024, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (attn): Linear(in_features=128, out_features=1, bias=True)
  (cls): Linear(in_features=128, out_features=1, bias=True)
)

[FA_PT_k=4] auc=0.806 balacc=0.743 acc=0.745 (n_test=51)

=== DEEP-ONLY Attention MIL 5-fold summary (mean ± std) ===
           auc    balacc       acc
mean  0.713999  0.608822  0.605560
std   0.124611  0.088990  0.097282


3 stream mil

In [116]:
# ==========================================
# 3-STREAM GATED ATTENTION MIL
# Deep features -> pooled representation -> classifier
# Concepts + Mitosis features -> contribute to attention logits only
#
# Inputs:
# - Deep .pt per slide: {"features": (N,D), "coords": (N,2)} in PT_FEAT_DIR
# - Concept CSV (patch-level): slide_id, x, y, concept_* in PATCH_CONCEPT_CSV
# - Mitosis CSV (patch-level): slide_id, patch_id, mito_* in MITO_CSV
#
# Alignment:
# - deep coords (x,y) matched to concept (x,y)
# - deep coords matched to mito via parsing patch_id "patch_*_x{X}_y{Y}"
# ==========================================

import os
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, accuracy_score

# =========================
# CONFIG
# =========================
PATCH_CONCEPT_CSV = r"C:\Users\Vivian\Documents\CONCH\test_text_encoder\slide_concept_scores\noCAP_patch_ptfile_notopk_conf_PATCH.csv"
MITO_CSV          = r"C:\Users\Vivian\Documents\dsmil-wsi\roboflow\patch_feats.csv"
PT_FEAT_DIR       = r"C:\Users\Vivian\Documents\CONCH\conch_img_feats\10x_feats\40x\pt"
CV_ROOT           = r"C:/Users/Vivian/Documents/PANTHER/PANTHER/src/splits/cross-val"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 0

SLIDE_COL = "slide_id"
LABEL_COL = "true_label_str"   # if missing, label derived from slide_id prefix FA/PT
X_COL, Y_COL = "x", "y"
CONCEPT_PREFIX = "concept_"

# Bag controls
BAG_CAP = None     # e.g., 4096 or None
TOPK    = None     # e.g., 1024 or None
TOPK_MODE = "deep_norm"  # "deep_norm" | "concept_score" | "mito_score"

# Train controls
EPOCHS = 8
LR = 1e-3
WEIGHT_DECAY = 1e-4
HID = 128
DROPOUT = 0.1

torch.manual_seed(SEED)
np.random.seed(SEED)

# =========================
# HELPERS
# =========================
def labels_to_int(arr):
    s = pd.Series(arr).astype(str).str.upper().str.strip()
    y = s.map({"FA": 0, "PT": 1})
    if y.isna().any():
        raise ValueError(f"Bad labels: {s[y.isna()].unique()}")
    return y.astype(int).to_numpy()

def derive_label_from_slide_id(slide_id: str) -> int:
    s = str(slide_id).upper().strip()
    if s.startswith("FA"):
        return 0
    if s.startswith("PT"):
        return 1
    raise ValueError(f"Can't derive label from slide_id='{slide_id}'")

def read_slide_list(csv_path: str):
    d = pd.read_csv(csv_path)
    if "slide" in d.columns:
        slides = d["slide"].astype(str).tolist()
    elif "slide_id" in d.columns:
        slides = d["slide_id"].astype(str).tolist()
    else:
        slides = d.iloc[:, 0].astype(str).tolist()
    return [os.path.splitext(s.strip())[0] for s in slides]

def find_fold_dirs(cv_root: str):
    out = []
    for name in sorted(os.listdir(cv_root)):
        p = os.path.join(cv_root, name)
        if os.path.isdir(p) and all(os.path.isfile(os.path.join(p, f"{x}.csv")) for x in ["train","val","test"]):
            out.append(p)
    return out

def load_slide_pt(pt_path: str):
    obj = torch.load(pt_path, map_location="cpu", weights_only=True)
    feats = obj["features"].float()  # (N,D)
    coords = obj["coords"].int()     # (N,2)
    return feats.numpy(), coords.numpy()

def eval_metrics(y_true, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(int)
    return {
        "auc": roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else np.nan,
        "balacc": balanced_accuracy_score(y_true, y_pred),
        "acc": accuracy_score(y_true, y_pred),
    }

# Parse patch_id: "patch_1018_x6720_y8288" -> (6720, 8288)
_PATCH_RE = re.compile(r"_x(?P<x>-?\d+)_y(?P<y>-?\d+)\b")
def parse_xy_from_patch_id(patch_id: str):
    m = _PATCH_RE.search(str(patch_id))
    if m is None:
        return None
    return int(m.group("x")), int(m.group("y"))

def make_train_scaler_stats(df: pd.DataFrame, cols):
    """Robust-ish scaler stats using median/IQR on TRAIN only."""
    X = df[cols].to_numpy(dtype=np.float32)
    med = np.nanmedian(X, axis=0)
    q1  = np.nanpercentile(X, 25, axis=0)
    q3  = np.nanpercentile(X, 75, axis=0)
    iqr = (q3 - q1)
    iqr[iqr < 1e-6] = 1.0
    return med.astype(np.float32), iqr.astype(np.float32)

def apply_train_scaler(X: np.ndarray, med: np.ndarray, iqr: np.ndarray):
    X = X.astype(np.float32)
    return (X - med) / iqr

# =========================
# DATASET
# =========================
class ThreeStreamBagDataset(Dataset):
    """
    Returns one slide (bag):
      Xd: (N, Ddeep)
      Xc: (N, Kconcept)
      Xm: (N, Mmito)
      y : scalar
    Alignment via (x,y) coords for concept and mito.
    """
    def __init__(
        self,
        df_concept: pd.DataFrame,
        df_mito: pd.DataFrame,
        slide_ids,
        concept_cols,
        mito_cols,
        pt_dir,
        mito_med=None,
        mito_iqr=None,
        bag_cap=None,
        topk=None,
        topk_mode="deep_norm",
        seed=0,
    ):
        self.df_concept = df_concept[df_concept[SLIDE_COL].isin(slide_ids)].copy()
        self.df_mito    = df_mito[df_mito[SLIDE_COL].isin(slide_ids)].copy()
        self.slide_ids  = sorted(list(set(self.df_concept[SLIDE_COL].unique()).intersection(set(slide_ids))))
        self.concept_cols = concept_cols
        self.mito_cols = mito_cols
        self.pt_dir = pt_dir

        self.mito_med = mito_med
        self.mito_iqr = mito_iqr

        self.bag_cap = bag_cap
        self.topk = topk
        self.topk_mode = topk_mode

        self.rng = np.random.default_rng(seed)

        # labels
        if LABEL_COL in self.df_concept.columns:
            lab = self.df_concept.groupby(SLIDE_COL)[LABEL_COL].first()
            self.y_map = {sid: int(labels_to_int([lab.loc[sid]])[0]) for sid in self.slide_ids}
        else:
            self.y_map = {sid: derive_label_from_slide_id(sid) for sid in self.slide_ids}

        # per-slide groups for faster maps
        self.by_slide_concept = {sid: d for sid, d in self.df_concept.groupby(SLIDE_COL)}
        self.by_slide_mito    = {sid: d for sid, d in self.df_mito.groupby(SLIDE_COL)} if len(self.df_mito) else {}

        # filter slides missing pt files
        keep = []
        for sid in self.slide_ids:
            if os.path.isfile(os.path.join(self.pt_dir, f"{sid}.pt")):
                keep.append(sid)
        self.slide_ids = keep

    def __len__(self):
        return len(self.slide_ids)

    def __getitem__(self, idx):
        sid = self.slide_ids[idx]
        pt_path = os.path.join(self.pt_dir, f"{sid}.pt")
        E, coords = load_slide_pt(pt_path)  # E:(N,D), coords:(N,2)
        Xd = E.astype(np.float32)
        coords = coords.astype(int)

        # --- concept map (x,y) -> concept vec
        dc = self.by_slide_concept[sid]
        c_keys = list(zip(dc[X_COL].astype(int).to_numpy(), dc[Y_COL].astype(int).to_numpy()))
        C = dc[self.concept_cols].to_numpy(dtype=np.float32)
        concept_map = {k: C[i] for i, k in enumerate(c_keys)}

        # --- mito map (x,y) -> mito vec (from patch_id parsing)
        mito_map = {}
        if sid in self.by_slide_mito:
            dm = self.by_slide_mito[sid]
            # build keys from patch_id
            xy = dm["patch_id"].map(parse_xy_from_patch_id)
            ok = xy.notna()
            if ok.any():
                xy2 = xy[ok].tolist()
                M = dm.loc[ok, self.mito_cols].to_numpy(dtype=np.float32)
                for i, k in enumerate(xy2):
                    mito_map[k] = M[i]

        # --- align: keep only deep patches that have BOTH concept and mito matches
        # If you want to allow missing mito, change required_mito to False.
        required_mito = True

        matched_idx = []
        Xc_list, Xm_list = [], []

        for i, (x, y) in enumerate(coords):
            cv = concept_map.get((int(x), int(y)), None)
            mv = mito_map.get((int(x), int(y)), None)
            if cv is None:
                continue
            if required_mito and mv is None:
                continue
            if mv is None:
                mv = np.zeros((len(self.mito_cols),), dtype=np.float32)
            matched_idx.append(i)
            Xc_list.append(cv)
            Xm_list.append(mv)

        if len(matched_idx) == 0:
            # fallback: keep 1 instance (avoid crash). You should log this.
            Xd = Xd[:1]
            Xc = np.zeros((1, len(self.concept_cols)), dtype=np.float32)
            Xm = np.zeros((1, len(self.mito_cols)), dtype=np.float32)
        else:
            Xd = Xd[matched_idx]
            Xc = np.stack(Xc_list, axis=0).astype(np.float32)
            Xm = np.stack(Xm_list, axis=0).astype(np.float32)

        # ---- mito normalization (train-fit stats passed in)
        # log1p for counts
        if "n_mitoses" in self.mito_cols:
            j = self.mito_cols.index("n_mitoses")
            Xm[:, j] = np.log1p(np.clip(Xm[:, j], 0, None))
        if "n_highconf" in self.mito_cols:
            j = self.mito_cols.index("n_highconf")
            Xm[:, j] = np.log1p(np.clip(Xm[:, j], 0, None))

        if self.mito_med is not None and self.mito_iqr is not None:
            Xm = apply_train_scaler(Xm, self.mito_med, self.mito_iqr)

        # ---- selection indices (apply consistently)
        N = Xd.shape[0]
        sel = np.arange(N)

        if self.bag_cap is not None and N > self.bag_cap:
            sel = self.rng.choice(N, size=self.bag_cap, replace=False)
            Xd, Xc, Xm = Xd[sel], Xc[sel], Xm[sel]

        if self.topk is not None and Xd.shape[0] > self.topk:
            if self.topk_mode == "deep_norm":
                score = np.linalg.norm(Xd, axis=1)
            elif self.topk_mode == "concept_score":
                score = Xc.mean(axis=1)
            elif self.topk_mode == "mito_score":
                # reasonable default: weight mitosis presence + counts
                pres = Xm[:, self.mito_cols.index("mitosis_present")] if "mitosis_present" in self.mito_cols else 0.0
                cnt  = Xm[:, self.mito_cols.index("n_mitoses")] if "n_mitoses" in self.mito_cols else 0.0
                score = pres + cnt
            else:
                raise ValueError(f"Bad topk_mode: {self.topk_mode}")

            keep = np.argsort(-score)[: self.topk]
            Xd, Xc, Xm = Xd[keep], Xc[keep], Xm[keep]

        y = self.y_map[sid]
        return (
            torch.from_numpy(Xd),
            torch.from_numpy(Xc),
            torch.from_numpy(Xm),
            torch.tensor(y, dtype=torch.long),
            sid
        )

def collate_bag(batch):
    return batch  # list of length B, each is one slide

# =========================
# MODEL
# =========================
class ThreeStreamGatedAttentionMIL(nn.Module):
    """
    Attention logits:
      a_i = attn_d(phi_d(Xd_i)) + attn_c(phi_c(Xc_i)) + attn_m(phi_m(Xm_i))

    Pooling uses deep stream only:
      z = sum softmax(a)_i * phi_d(Xd_i)
    """
    def __init__(self, deep_dim, concept_dim, mito_dim, hid=128, drop=0.1):
        super().__init__()
        self.phi_d = nn.Sequential(nn.Linear(deep_dim, hid), nn.ReLU(), nn.Dropout(drop))
        self.phi_c = nn.Sequential(nn.Linear(concept_dim, hid), nn.ReLU(), nn.Dropout(drop))
        self.phi_m = nn.Sequential(nn.Linear(mito_dim, hid), nn.ReLU(), nn.Dropout(drop))

        self.attn_d = nn.Linear(hid, 1)
        self.attn_c = nn.Linear(hid, 1)
        self.attn_m = nn.Linear(hid, 1)

        self.cls = nn.Linear(hid, 1)
        self.alpha_m = nn.Parameter(torch.tensor(0.0)) # optional learnable weight for mito attention

    def forward(self, Xd, Xc, Xm):
        Hd = self.phi_d(Xd)                 # (N,hid)
        Hc = self.phi_c(Xc)                 # (N,hid)
        Hm = self.phi_m(Xm)                 # (N,hid)

        # a = self.attn_d(Hd).squeeze(1) + self.attn_c(Hc).squeeze(1) + self.attn_m(Hm).squeeze(1)
        a = self.attn_d(Hd).squeeze(1) + self.attn_c(Hc).squeeze(1) + self.alpha_m * self.attn_m(Hm).squeeze(1) # optional weighting of mito attention
        w = torch.softmax(a, dim=0)         # (N,)

        z = (w.unsqueeze(1) * Hd).sum(0)    # (hid,)
        logit = self.cls(z).squeeze(0)      # ()
        return logit, w

# =========================
# LOAD TABLES
# =========================
df_con = pd.read_csv(PATCH_CONCEPT_CSV)
df_con.columns = [c.strip() for c in df_con.columns]
for col in [SLIDE_COL, X_COL, Y_COL]:
    if col not in df_con.columns:
        raise ValueError(f"Missing '{col}' in concepts CSV")

concept_cols = [c for c in df_con.columns if c.startswith(CONCEPT_PREFIX)]
if not concept_cols:
    raise ValueError("No concept_* columns found in concepts CSV")

# numeric + impute
df_con[concept_cols] = df_con[concept_cols].apply(pd.to_numeric, errors="coerce")
df_con[concept_cols] = df_con[concept_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)

df_mito = pd.read_csv(MITO_CSV)

# --- normalize mito slide_id to match deep/concept slide IDs ---
# Example: "FA 56B_mitosis_detections" -> "FA 56B"
df_mito[SLIDE_COL] = (
    df_mito[SLIDE_COL]
      .astype(str)
      .str.strip()
      .str.replace(r"_mitosis_detections$", "", regex=True)
      .str.replace(r"_mitosis_detection$", "", regex=True)
      .str.replace(r"\s+", " ", regex=True)   # collapse double spaces
)

# (optional) if your deep/concept slide_ids never have file extensions:
df_mito[SLIDE_COL] = df_mito[SLIDE_COL].str.replace(r"\.svs$|\.tif$|\.tiff$|\.mrxs$|\.ndpi$|\.vsi$", "", regex=True)

# --- ensure mito numeric columns are clean ---
mito_cols = ["n_mitoses","n_highconf","conf_mean","conf_max","highconf_fraction","mitosis_present"]
df_mito[mito_cols] = df_mito[mito_cols].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)

# quick sanity check (should be >0)
print("Unique mito slides after normalization:", df_mito[SLIDE_COL].nunique())
print("Example mito slide IDs:", df_mito[SLIDE_COL].dropna().unique()[:5])

df_mito.columns = [c.strip() for c in df_mito.columns]
for col in [SLIDE_COL, "patch_id"]:
    if col not in df_mito.columns:
        raise ValueError(f"Missing '{col}' in mito CSV")

mito_cols = ["n_mitoses","n_highconf","conf_mean","conf_max","highconf_fraction","mitosis_present"]
for c in mito_cols:
    if c not in df_mito.columns:
        raise ValueError(f"Missing '{c}' in mito CSV")

df_mito[mito_cols] = df_mito[mito_cols].apply(pd.to_numeric, errors="coerce")
df_mito[mito_cols] = df_mito[mito_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)

fold_dirs = find_fold_dirs(CV_ROOT)

# =========================
# 5-FOLD CV
# =========================
all_fold = []

for fold_dir in fold_dirs:
    fold_name = os.path.basename(fold_dir)

    train_slides = read_slide_list(os.path.join(fold_dir, "train.csv"))
    val_slides   = read_slide_list(os.path.join(fold_dir, "val.csv"))
    test_slides  = read_slide_list(os.path.join(fold_dir, "test.csv"))
    fit_slides   = train_slides + val_slides

    # ---- fit mito scaler on TRAIN+VAL slides only ----
    df_mito_fit = df_mito[df_mito[SLIDE_COL].isin(fit_slides)].copy()

    # Apply log1p here before computing median/IQR so scaling matches dataset-time transforms
    df_mito_fit["n_mitoses"]  = np.log1p(np.clip(df_mito_fit["n_mitoses"].to_numpy(dtype=np.float32), 0, None))
    df_mito_fit["n_highconf"] = np.log1p(np.clip(df_mito_fit["n_highconf"].to_numpy(dtype=np.float32), 0, None))

    mito_med, mito_iqr = make_train_scaler_stats(df_mito_fit, mito_cols)

    ds_tr = ThreeStreamBagDataset(
        df_concept=df_con,
        df_mito=df_mito,
        slide_ids=fit_slides,
        concept_cols=concept_cols,
        mito_cols=mito_cols,
        pt_dir=PT_FEAT_DIR,
        mito_med=mito_med,
        mito_iqr=mito_iqr,
        bag_cap=BAG_CAP,
        topk=TOPK,
        topk_mode=TOPK_MODE,
        seed=SEED,
    )
    ds_te = ThreeStreamBagDataset(
        df_concept=df_con,
        df_mito=df_mito,
        slide_ids=test_slides,
        concept_cols=concept_cols,
        mito_cols=mito_cols,
        pt_dir=PT_FEAT_DIR,
        mito_med=mito_med,
        mito_iqr=mito_iqr,
        bag_cap=BAG_CAP,
        topk=TOPK,
        topk_mode=TOPK_MODE,
        seed=SEED,
    )

    dl_tr = DataLoader(ds_tr, batch_size=1, shuffle=True, collate_fn=collate_bag)
    dl_te = DataLoader(ds_te, batch_size=1, shuffle=False, collate_fn=collate_bag)

    if len(ds_tr) == 0 or len(ds_te) == 0:
        print(f"[{fold_name}] skipped (empty train/test after filtering)")
        continue

    # infer dims
    Xd0, Xc0, Xm0, _, _ = ds_tr[0]
    model = ThreeStreamGatedAttentionMIL(
        deep_dim=Xd0.shape[1],
        concept_dim=Xc0.shape[1],
        mito_dim=Xm0.shape[1],
        hid=HID,
        drop=DROPOUT
    ).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.BCEWithLogitsLoss()

    # train
    model.train()
    for ep in range(EPOCHS):
        for batch in dl_tr:
            (Xd, Xc, Xm, y, sid) = batch[0]
            Xd = Xd.to(DEVICE)
            Xc = Xc.to(DEVICE)
            Xm = Xm.to(DEVICE)
            y  = y.float().to(DEVICE)

            logit, _w = model(Xd, Xc, Xm)
            loss = loss_fn(logit.view(1), y.view(1))

            opt.zero_grad()
            loss.backward()
            opt.step()

    # eval
    model.eval()
    y_true, y_prob = [], []
    with torch.no_grad():
        for batch in dl_te:
            (Xd, Xc, Xm, y, sid) = batch[0]
            Xd = Xd.to(DEVICE)
            Xc = Xc.to(DEVICE)
            Xm = Xm.to(DEVICE)

            logit, _w = model(Xd, Xc, Xm)
            prob = torch.sigmoid(logit).item()

            y_true.append(int(y.item()))
            y_prob.append(prob)

    y_true = np.array(y_true)
    y_prob = np.array(y_prob)
    m = eval_metrics(y_true, y_prob, thr=0.5)

    all_fold.append({"fold": fold_name, **m, "n_test": len(y_true)})
    print(f"[{fold_name}] auc={m['auc']:.3f} balacc={m['balacc']:.3f} acc={m['acc']:.3f} (n_test={len(y_true)})")

res = pd.DataFrame(all_fold)
print("\n=== 3-STREAM GATED MIL 5-fold summary (mean ± std) ===")
print(res[["auc","balacc","acc"]].agg(["mean","std"]))


Unique mito slides after normalization: 240
Example mito slide IDs: ['FA 56B' 'FA 57B' 'FA 58B' 'FA 59B' 'FA 60 B']


ThreeStreamGatedAttentionMIL(
  (phi_d): Sequential(
    (0): Linear(in_features=512, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (phi_c): Sequential(
    (0): Linear(in_features=50, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (phi_m): Sequential(
    (0): Linear(in_features=6, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (attn_d): Linear(in_features=128, out_features=1, bias=True)
  (attn_c): Linear(in_features=128, out_features=1, bias=True)
  (attn_m): Linear(in_features=128, out_features=1, bias=True)
  (cls): Linear(in_features=128, out_features=1, bias=True)
)

ThreeStreamGatedAttentionMIL(
  (phi_d): Sequential(
    (0): Linear(in_features=512, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (phi_c): Sequential(
    (0): Linear(in_features=50, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (phi_m): Sequential(
    (0): Linear(in_features=6, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (attn_d): Linear(in_features=128, out_features=1, bias=True)
  (attn_c): Linear(in_features=128, out_features=1, bias=True)
  (attn_m): Linear(in_features=128, out_features=1, bias=True)
  (cls): Linear(in_features=128, out_features=1, bias=True)
)

[FA_PT_k=0] auc=0.818 balacc=0.706 acc=0.702 (n_test=47)


ThreeStreamGatedAttentionMIL(
  (phi_d): Sequential(
    (0): Linear(in_features=512, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (phi_c): Sequential(
    (0): Linear(in_features=50, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (phi_m): Sequential(
    (0): Linear(in_features=6, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (attn_d): Linear(in_features=128, out_features=1, bias=True)
  (attn_c): Linear(in_features=128, out_features=1, bias=True)
  (attn_m): Linear(in_features=128, out_features=1, bias=True)
  (cls): Linear(in_features=128, out_features=1, bias=True)
)

...

ThreeStreamGatedAttentionMIL(
  (phi_d): Sequential(
    (0): Linear(in_features=512, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (phi_c): Sequential(
    (0): Linear(in_features=50, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (phi_m): Sequential(
    (0): Linear(in_features=6, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (attn_d): Linear(in_features=128, out_features=1, bias=True)
  (attn_c): Linear(in_features=128, out_features=1, bias=True)
  (attn_m): Linear(in_features=128, out_features=1, bias=True)
  (cls): Linear(in_features=128, out_features=1, bias=True)
)

[FA_PT_k=3] auc=0.853 balacc=0.775 acc=0.776 (n_test=49)


ThreeStreamGatedAttentionMIL(
  (phi_d): Sequential(
    (0): Linear(in_features=512, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (phi_c): Sequential(
    (0): Linear(in_features=50, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (phi_m): Sequential(
    (0): Linear(in_features=6, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (attn_d): Linear(in_features=128, out_features=1, bias=True)
  (attn_c): Linear(in_features=128, out_features=1, bias=True)
  (attn_m): Linear(in_features=128, out_features=1, bias=True)
  (cls): Linear(in_features=128, out_features=1, bias=True)
)

ThreeStreamGatedAttentionMIL(
  (phi_d): Sequential(
    (0): Linear(in_features=512, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (phi_c): Sequential(
    (0): Linear(in_features=50, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (phi_m): Sequential(
    (0): Linear(in_features=6, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (attn_d): Linear(in_features=128, out_features=1, bias=True)
  (attn_c): Linear(in_features=128, out_features=1, bias=True)
  (attn_m): Linear(in_features=128, out_features=1, bias=True)
  (cls): Linear(in_features=128, out_features=1, bias=True)
)

[FA_PT_k=4] auc=0.682 balacc=0.665 acc=0.667 (n_test=51)

=== 3-STREAM GATED MIL 5-fold summary (mean ± std) ===
           auc    balacc       acc
mean  0.808211  0.705828  0.708861
std   0.072068  0.047787  0.046495


analyze/classify concept scores

In [109]:
import pandas as pd
import numpy as np

# ---- CONFIG ----
CSV_PATH = r"C:\Users\Vivian\Documents\CONCH\test_text_encoder\slide_concept_scores\noCAP_patch_ptfile_notopk_conf.csv"  # <-- change
LABEL_COL = "true_label_str"   # FA/PT
SLIDE_COL = "slide_id"

# ---- LOAD ----
df = pd.read_csv(CSV_PATH)
df.columns = [c.strip() for c in df.columns]
df[LABEL_COL] = df[LABEL_COL].astype(str).str.upper().str.strip()
df = df[df[LABEL_COL].isin(["FA", "PT"])].copy()
df["y_pt"] = (df[LABEL_COL] == "PT").astype(int)

print("Rows:", len(df), "Slides:", df[SLIDE_COL].nunique() if SLIDE_COL in df.columns else "N/A")
print(df[LABEL_COL].value_counts())

# ---- FIND FEATURE COLUMNS ----
bucket_cols = [c for c in df.columns if c.startswith("bucket_")]
concept_cols = [c for c in df.columns if c.startswith("concept_")]

print("# bucket cols:", len(bucket_cols))
print("# concept cols:", len(concept_cols))

# ============================================================
# 1) FA vs PT means (buckets)
# ============================================================
key_buckets = [c for c in
               ["bucket_PT_lean", "bucket_mitosis", "bucket_FA_lean", "bucket_NORMAL_lean", "bucket_controls"]
               if c in df.columns]

if key_buckets:
    print("\n=== FA vs PT mean bucket scores ===")
    means = df.groupby(LABEL_COL)[key_buckets].mean()
    stds  = df.groupby(LABEL_COL)[key_buckets].std()
    print("MEAN:")
    display(means)
    print("STD:")
    display(stds)
else:
    print("\nNo key bucket columns found in CSV.")

# ============================================================
# 2) Rank concepts by correlation with PT label (0/1)
#    (Pearson on 0/1 label == point-biserial correlation)
# ============================================================
def rank_by_corr(cols):
    out = []
    y = df["y_pt"].to_numpy()
    for c in cols:
        x = pd.to_numeric(df[c], errors="coerce").to_numpy()
        m = np.isfinite(x)
        if m.sum() < 5:
            continue
        r = np.corrcoef(x[m], y[m])[0, 1]
        out.append((c, float(r), int(m.sum()),
                    float(np.nanmean(pd.to_numeric(df.loc[df["y_pt"] == 0, c], errors="coerce"))),
                    float(np.nanmean(pd.to_numeric(df.loc[df["y_pt"] == 1, c], errors="coerce")))))
    res = pd.DataFrame(out, columns=["feature", "corr_with_PT", "n", "mean_FA", "mean_PT"])
    return res.sort_values("corr_with_PT", ascending=False)

if concept_cols:
    concept_corr = rank_by_corr(concept_cols)
    print("\n=== Top concepts correlated with PT (higher = more PT-like) ===")
    display(concept_corr.head(20))
    print("\n=== Top concepts correlated with FA (more negative = more FA-like) ===")
    display(concept_corr.tail(20))
else:
    print("\nNo concept_* columns found (did you save per-concept slide means?).")

# ============================================================
# 3) Rank concepts by mean difference (mean_PT - mean_FA)
# ============================================================
def rank_by_mean_diff(cols):
    out = []
    for c in cols:
        x = pd.to_numeric(df[c], errors="coerce")
        fa = x[df[LABEL_COL] == "FA"]
        pt = x[df[LABEL_COL] == "PT"]
        if fa.notna().sum() < 5 or pt.notna().sum() < 5:
            continue
        out.append((c, float(pt.mean() - fa.mean()),
                    float(fa.mean()), float(pt.mean()),
                    int(fa.notna().sum()), int(pt.notna().sum())))
    res = pd.DataFrame(out, columns=["feature", "PT_minus_FA", "mean_FA", "mean_PT", "n_FA", "n_PT"])
    return res.sort_values("PT_minus_FA", ascending=False)

if concept_cols:
    concept_diff = rank_by_mean_diff(concept_cols)
    print("\n=== Concepts ranked by (mean_PT - mean_FA) ===")
    display(concept_diff.head(20))
    display(concept_diff.tail(20))


Rows: 240 Slides: 240
true_label_str
FA    121
PT    119
Name: count, dtype: int64
# bucket cols: 8
# concept cols: 50

=== FA vs PT mean bucket scores ===
MEAN:


,bucket_PT_lean,bucket_mitosis,bucket_FA_lean,bucket_NORMAL_lean,bucket_controls
true_label_str,,,,,
FA,0.460745,0.415011,0.450234,0.476379,0.440188
PT,0.475342,0.455558,0.472736,0.473956,0.433213


STD:


,bucket_PT_lean,bucket_mitosis,bucket_FA_lean,bucket_NORMAL_lean,bucket_controls
true_label_str,,,,,
FA,0.022816,0.042605,0.028744,0.018830,0.024122
PT,0.019676,0.040097,0.023619,0.018716,0.025592



=== Top concepts correlated with PT (higher = more PT-like) ===


...


=== Top concepts correlated with FA (more negative = more FA-like) ===


,feature,corr_with_PT,n,mean_FA,mean_PT
42,concept_42,0.089273,240,0.390874,0.397158
30,concept_30,0.069834,240,0.377494,0.383154
29,concept_29,0.069825,240,0.048621,0.052855
15,concept_15,0.051813,240,0.606606,0.609612
20,concept_20,0.045729,240,0.366477,0.368848
18,concept_18,0.037371,240,0.510339,0.512672
17,concept_17,0.032151,240,0.594714,0.596861
45,concept_45,0.028430,240,0.447314,0.448965
25,concept_25,0.012506,240,0.423913,0.424962
46,concept_46,0.009613,240,0.564515,0.565237



=== Concepts ranked by (mean_PT - mean_FA) ===


,feature,PT_minus_FA,mean_FA,mean_PT,n_FA,n_PT
40,concept_40,0.059507,0.339746,0.399253,121,119
36,concept_36,0.055011,0.374973,0.429984,121,119
2,concept_2,0.046753,0.356658,0.403411,121,119
27,concept_27,0.043922,0.398368,0.442289,121,119
12,concept_12,0.040929,0.384887,0.425816,121,119
34,concept_34,0.040388,0.594267,0.634656,121,119
4,concept_4,0.038186,0.657927,0.696113,121,119
24,concept_24,0.038026,0.551342,0.589368,121,119
35,concept_35,0.037988,0.456869,0.494857,121,119
22,concept_22,0.037641,0.477840,0.515480,121,119


,feature,PT_minus_FA,mean_FA,mean_PT,n_FA,n_PT
42,concept_42,0.006284,0.390874,0.397158,121,119
30,concept_30,0.005661,0.377494,0.383154,121,119
29,concept_29,0.004234,0.048621,0.052855,121,119
15,concept_15,0.003006,0.606606,0.609612,121,119
20,concept_20,0.002371,0.366477,0.368848,121,119
18,concept_18,0.002332,0.510339,0.512672,121,119
17,concept_17,0.002146,0.594714,0.596861,121,119
45,concept_45,0.001651,0.447314,0.448965,121,119
25,concept_25,0.001049,0.423913,0.424962,121,119
46,concept_46,0.000722,0.564515,0.565237,121,119


In [117]:
import os
import glob
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, accuracy_score, confusion_matrix

# ============================================================
# CONFIG (EDIT THESE)
# ============================================================
# CONCH_CSVS = [
#     r"C:\Users\Vivian\Documents\CONCH\test_text_encoder\concept_scores_v2\concept_score_20x_randN800_topk50_conf.csv",
#     r"C:\Users\Vivian\Documents\CONCH\test_text_encoder\concept_scores_v2\concept_score_40x_randN800_topk50_conf.csv"
# ]
CONCH_CSVS = [r'C:\Users\Vivian\Documents\CONCH\test_text_encoder\slide_concept_scores\noCAP_patch_ptfile_notopk_conf_PATCH.csv']
CV_ROOT = r"C:\Users\Vivian\Documents\PANTHER\PANTHER\src\splits\cross-val"   # contains fold dirs with train.csv/val.csv/test.csv
OUT_DIR = r"C:\Users\Vivian\Documents\CONCH\test_text_encoder\slide_concept_scores\concept_clf_results_10x"

FEATURE_MODE = "concept"   # "bucket" | "concept" | "both"
USE_TRAIN_PLUS_VAL = True  # match your earlier setup
SEED = 0

# Logistic settings:
# - For "concept": L1 helps avoid overfit / selects concepts
# - For "bucket": L2 is fine
PENALTY = "l1" if FEATURE_MODE in ("concept", "both") else "l2"
C_VAL = 1.0  # regularization strength; smaller => more regularization (try 0.1, 0.3, 1.0)

os.makedirs(OUT_DIR, exist_ok=True)

# ============================================================
# Helpers
# ============================================================
def normalize_slide_id(s: str) -> str:
    s = str(s).strip()
    s = os.path.splitext(s)[0]
    return s

def read_slide_list(csv_path: str):
    d = pd.read_csv(csv_path)
    if "slide" in d.columns:
        slides = d["slide"].astype(str).tolist()
    elif "slide_id" in d.columns:
        slides = d["slide_id"].astype(str).tolist()
    else:
        slides = d.iloc[:, 0].astype(str).tolist()
    return [normalize_slide_id(x) for x in slides]

def labels_to_int(y_str: pd.Series) -> np.ndarray:
    y = y_str.astype(str).str.upper().str.strip().map({"FA": 0, "PT": 1})
    if y.isna().any():
        bad = y_str[y.isna()].unique()
        raise ValueError(f"Unexpected labels: {bad} (expected FA/PT)")
    return y.astype(int).to_numpy()

def find_fold_dirs(cv_root: str):
    # robust: any subdir containing train/val/test.csv
    out = []
    for d in sorted(glob.glob(os.path.join(cv_root, "*"))):
        if os.path.isdir(d) and all(os.path.exists(os.path.join(d, f)) for f in ["train.csv", "val.csv", "test.csv"]):
            out.append(d)
    return out

def eval_prob(y_true, prob, thr=0.5):
    pred = (prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {
        "auc": roc_auc_score(y_true, prob) if len(np.unique(y_true)) == 2 else np.nan,
        "balacc": balanced_accuracy_score(y_true, pred),
        "acc": accuracy_score(y_true, pred),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

def get_feature_cols(df: pd.DataFrame, mode: str):
    cols = []
    if mode in ("bucket", "both"):
        cols += [c for c in df.columns if str(c).startswith("bucket_")]
    if mode in ("concept", "both"):
        cols += [c for c in df.columns if str(c).startswith("concept_")]
    if not cols:
        raise ValueError(f"No feature columns found for mode={mode}. "
                         f"Found bucket_={any(str(c).startswith('bucket_') for c in df.columns)}, "
                         f"concept_={any(str(c).startswith('concept_') for c in df.columns)}")
    return cols

# ============================================================
# Load + combine CONCH CSVs
# ============================================================
dfs = [pd.read_csv(p) for p in CONCH_CSVS]
df_all = pd.concat(dfs, ignore_index=True)
df_all.columns = [str(c).strip() for c in df_all.columns]

# Ensure one row per slide_id (your file should already be one row per slide;
# but this keeps it robust if you concatenate multiple CSVs)
SLIDE_COL = "slide_id"
LABEL_COL = "true_label_str"

if SLIDE_COL not in df_all.columns or LABEL_COL not in df_all.columns:
    raise ValueError(f"CSV must contain {SLIDE_COL} and {LABEL_COL}. Columns: {list(df_all.columns)[:30]}")

df_all[SLIDE_COL] = df_all[SLIDE_COL].astype(str).map(normalize_slide_id)
df_all[LABEL_COL] = df_all[LABEL_COL].astype(str).str.upper().str.strip()

# Pick features
feat_cols = get_feature_cols(df_all, FEATURE_MODE)

# Coerce to numeric
for c in feat_cols:
    df_all[c] = pd.to_numeric(df_all[c], errors="coerce")

# Aggregate to one row per slide (mean), keep first label/mag
agg = {c: "mean" for c in feat_cols}
agg[LABEL_COL] = "first"
if "mag" in df_all.columns:
    agg["mag"] = "first"

df_slide = df_all.groupby(SLIDE_COL, as_index=False).agg(agg)

# Clean NaNs/infs (median impute)
df_slide[feat_cols] = df_slide[feat_cols].replace([np.inf, -np.inf], np.nan)
df_slide[feat_cols] = df_slide[feat_cols].fillna(df_slide[feat_cols].median(numeric_only=True))

df_slide = df_slide.set_index(SLIDE_COL, drop=False)

print("Slides in CONCH table:", len(df_slide))
print("Feature mode:", FEATURE_MODE, "| #features:", len(feat_cols))
print(df_slide[LABEL_COL].value_counts())

# ============================================================
# Run CV folds
# ============================================================
fold_dirs = find_fold_dirs(CV_ROOT)
if not fold_dirs:
    raise ValueError(f"No fold dirs found under {CV_ROOT} containing train/val/test.csv")

fold_rows = []
coef_rows = []
pred_rows = []

for fold_dir in fold_dirs:
    fold_name = os.path.basename(fold_dir)

    train_slides = read_slide_list(os.path.join(fold_dir, "train.csv"))
    val_slides   = read_slide_list(os.path.join(fold_dir, "val.csv"))
    test_slides  = read_slide_list(os.path.join(fold_dir, "test.csv"))

    fit_slides = train_slides + (val_slides if USE_TRAIN_PLUS_VAL else [])
    fit_slides = list(dict.fromkeys(fit_slides))  # unique

    fit_in  = [s for s in fit_slides  if s in df_slide.index]
    test_in = [s for s in test_slides if s in df_slide.index]

    missing_fit  = len(fit_slides) - len(fit_in)
    missing_test = len(test_slides) - len(test_in)

    if len(test_in) == 0 or len(fit_in) == 0:
        print(f"[{fold_name}] skip (fit={len(fit_in)} test={len(test_in)})")
        continue

    Xtr = df_slide.loc[fit_in, feat_cols].to_numpy()
    ytr = labels_to_int(df_slide.loc[fit_in, LABEL_COL])

    Xte = df_slide.loc[test_in, feat_cols].to_numpy()
    yte = labels_to_int(df_slide.loc[test_in, LABEL_COL])

    # Standardize (important for concept_* / many features)
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(Xtr)
    Xte = scaler.transform(Xte)

    clf = LogisticRegression(
        penalty=PENALTY,
        C=C_VAL,
        solver="liblinear" if PENALTY == "l1" else "lbfgs",
        class_weight="balanced",
        random_state=SEED,
        max_iter=5000,
    )
    clf.fit(Xtr, ytr)
    prob = clf.predict_proba(Xte)[:, 1]

    m = eval_prob(yte, prob, thr=0.5)
    fold_rows.append({
        "fold": fold_name,
        "n_train": len(fit_in),
        "n_test": len(test_in),
        "n_features": len(feat_cols),
        "missing_fit": missing_fit,
        "missing_test": missing_test,
        "penalty": PENALTY,
        "C": C_VAL,
        **m,
    })

    # Save coefficients (optional; can be wide if many concepts)
    coef_row = {"fold": fold_name}
    for c, w in zip(feat_cols, clf.coef_.ravel()):
        coef_row[c] = float(w)
    coef_rows.append(coef_row)

    # Save per-slide test preds
    pred_rows.append(pd.DataFrame({
        "fold": fold_name,
        "slide_id": test_in,
        "true_label": df_slide.loc[test_in, LABEL_COL].values,
        "prob_pt": prob,
        "pred_label": np.where(prob >= 0.5, "PT", "FA"),
    }))

    print(f"[{fold_name}] AUC={m['auc']:.3f} BalAcc={m['balacc']:.3f} Acc={m['acc']:.3f} "
          f"(n_test={len(test_in)} missing_test={missing_test})")

results = pd.DataFrame(fold_rows)
results_path = os.path.join(OUT_DIR, f"fold_metrics_{FEATURE_MODE}.csv")
# results.to_csv(results_path, index=False)
# print("\nSaved:", results_path)

summary = results[["auc", "balacc", "acc"]].agg(["mean", "std"])
print("\n=== Summary (mean ± std) ===")
display(summary)

coef_df = pd.DataFrame(coef_rows)
coef_path = os.path.join(OUT_DIR, f"coef_all_folds_{FEATURE_MODE}.csv")
coef_df.to_csv(coef_path, index=False)
print("Saved:", coef_path)

pred_df = pd.concat(pred_rows, ignore_index=True) if pred_rows else pd.DataFrame()
pred_path = os.path.join(OUT_DIR, f"test_preds_{FEATURE_MODE}.csv")
pred_df.to_csv(pred_path, index=False)
print("Saved:", pred_path)

# Optional: show top coefficients averaged across folds (most important features)
if not coef_df.empty:
    coef_only = coef_df.drop(columns=["fold"], errors="ignore")
    coef_summary = coef_only.agg(["mean", "std"]).T
    coef_summary["abs_mean"] = coef_summary["mean"].abs()
    coef_summary = coef_summary.sort_values("abs_mean", ascending=False)
    print("\n=== Top features by |mean coef| (across folds) ===")
    display(coef_summary.head(25))


Slides in CONCH table: 240
Feature mode: concept | #features: 50
true_label_str
FA    121
PT    119
Name: count, dtype: int64


LogisticRegression(class_weight='balanced', max_iter=5000, penalty='l1',
                   random_state=0, solver='liblinear')

[FA_PT_k=0] AUC=0.782 BalAcc=0.686 Acc=0.681 (n_test=47 missing_test=0)


LogisticRegression(class_weight='balanced', max_iter=5000, penalty='l1',
                   random_state=0, solver='liblinear')

[FA_PT_k=1] AUC=0.807 BalAcc=0.686 Acc=0.688 (n_test=48 missing_test=0)


...

LogisticRegression(class_weight='balanced', max_iter=5000, penalty='l1',
                   random_state=0, solver='liblinear')

[FA_PT_k=4] AUC=0.714 BalAcc=0.705 Acc=0.706 (n_test=51 missing_test=0)

=== Summary (mean ± std) ===


,auc,balacc,acc
mean,0.806791,0.716900,0.717341
std,0.063861,0.037444,0.039708


Saved: C:\Users\Vivian\Documents\CONCH\test_text_encoder\slide_concept_scores\concept_clf_results_10x\coef_all_folds_concept.csv
Saved: C:\Users\Vivian\Documents\CONCH\test_text_encoder\slide_concept_scores\concept_clf_results_10x\test_preds_concept.csv

=== Top features by |mean coef| (across folds) ===


,mean,std,abs_mean
concept_40,1.033127,0.390276,1.033127
concept_24,0.854767,0.472029,0.854767
concept_43,-0.691574,0.327463,0.691574
concept_17,-0.599609,0.317198,0.599609
concept_23,0.594443,0.759205,0.594443
concept_47,0.577281,0.481086,0.577281
concept_3,-0.300644,0.244050,0.300644
concept_16,-0.294989,0.192415,0.294989
concept_13,0.254340,0.312991,0.254340
concept_49,-0.244716,0.178727,0.244716
